In [1]:
%cd /app

/app


In [2]:
import argparse
import os
import sys

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import torch
torch.multiprocessing.set_start_method('spawn')

import jax
from lob.encoding import Vocab, Message_Tokenizer

from lob import inference_no_errcorr as inference
from lob.init_train import init_train_state, load_checkpoint, load_metadata, load_args_from_checkpoint

from lob import inference_no_errcorr as inference
import lob.encoding as encoding
import preproc as preproc

import jax.numpy as jnp
import numpy as np

from pathlib import Path
import os

import pandas as pd

import pandas as pd
import plotly.graph_objs as go
import yaml
import pickle

from filtration_utils import summary_table, build_zero_padded_series, plot_midprice_series_with_insertions, prepare_volatility_filtered_series, plot_midprice_series_with_mean_std
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from typing import Callable, Tuple, Optional, List, Dict

import ipywidgets as widgets
from IPython.display import display, clear_output

CUDA backend failed to initialize: jaxlib/cuda/versions_helpers.cc:98: operation cuInit(0) failed: CUDA_ERROR_NO_DEVICE (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)
/opt/conda/envs/myenv/lib/python3.12/site-packages/torch/cuda/__init__.py:619: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
2025-10-06 22:03:32.623448: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
# ===== Compare 3 approaches: hist / heur / gen for multiple volumes =====

# --- Define experiment configs for each volume ---
# You can fill in the experiment names for vol2 and vol3 below

volume_experiments = {
    "vol1": {  # 485
        "label": "V=485",
        "color": "blue",
        "approaches": {
            "hist": {
                "buy": 'exp_143_20251002_233705_hist_buy_485_b0_b7',
                "sell": 'exp_144_20251002_234327_hist_sell_485_b0_b7',
            },
            "heur": {
                "buy": 'exp_137_20251002_210350_heur_buy_485_b0_b7',
                "sell": 'exp_138_20251002_210917_heur_sell_485_b0_b7',
            },
            "gen": {
                "buy": 'exp_133_20250923_202808_gen_buy_485_b0_b7',
                "sell": 'exp_134_20250924_005646_gen_sell_485_b0_b7',
            }
        }
    },
    "vol2": {  # <--- FILL IN YOUR EXPERIMENTS FOR 2nd VOLUME
        "label": "V=300",  # e.g. "V=1000"
        "color": "green",
        "approaches": {
            "hist": {
                "buy": 'exp_141_20251002_232519_hist_buy_300_b0_b7',
                "sell": 'exp_142_20251002_233013_hist_sell_300_b0_b7',
            },
            "heur": {
                "buy": 'exp_135_20251002_171016_heur_buy_300_b0_b7',
                "sell": 'exp_136_20251002_201550_heur_sell_300_b0_b7',
            },
            "gen": {
                "buy": 'exp_131_20250923_104218_gen_buy_300_b0_b7',
                "sell": 'exp_132_20250923_154415_gen_sell_300_b0_b7',
            }
        }
    },
    "vol3": {  # <--- FILL IN YOUR EXPERIMENTS FOR 3rd VOLUME
        "label": "V=75",  # e.g. "V=2000"
        "color": "red",
        "approaches": {
            "hist": {
                "buy": 'exp_151_20251006_133342_hist_buy_75_b0_b7',
                "sell": 'exp_152_20251006_143539_hist_sell_75_b0_b7',
            },
            "heur": {
                "buy": 'exp_149_20251006_132251_heur_buy_75_b0_b7',
                "sell": 'exp_150_20251006_132807_heur_sell_75_b0_b7',
            },
            "gen": {
                "buy": 'exp_147_20251005_220723_gen_buy_75_b0_b7',
                "sell": 'exp_148_20251006_023201_gen_sell_75_b0_b7',
            }
        }
    }
}

# --- Line style for each approach ---
approach_linestyle = {
    "hist": "dot",
    "heur": "dash",
    "gen": "solid"
}

approach_labels = {
    "hist": "Hist",
    "heur": "Heur",
    "gen": "Gen"
}

sample_day_map = pd.read_csv(f'/app/batches_equal_sample_day_map.csv')

# --- Load and process data for all volumes and approaches ---
all_returns_stats = {}

for vol_key, vol_cfg in volume_experiments.items():
    all_returns_stats[vol_key] = {}
    for approach, exp in vol_cfg["approaches"].items():
        # Load sample indices after book filtration
        try:
            merged_buy_indx = pd.read_csv(f'/app/data_saved/{exp["buy"]}/samples_after_book_filtration.csv')
            merged_sell_indx = pd.read_csv(f'/app/data_saved/{exp["sell"]}/samples_after_book_filtration.csv')
        except Exception as e:
            print(f"Skip {vol_key}-{approach}: {e}")
            continue
        buy_sample_indices = set(merged_buy_indx['samples after book filtration'])
        sell_sample_indices = set(merged_sell_indx['samples after book filtration'])
        common_sample_indices_set = buy_sample_indices & sell_sample_indices

        # Load full filtered_merged.pkl for buy/sell
        try:
            with open(f'/app/data_saved/{exp["buy"]}/filtered_merged.pkl', 'rb') as f:
                merged_buy_full = pickle.load(f)
            with open(f'/app/data_saved/{exp["sell"]}/filtered_merged.pkl', 'rb') as f:
                merged_sell_full = pickle.load(f)
        except Exception as e:
            print(f"Skip {vol_key}-{approach}: {e}")
            continue

        # Filter to only common sample indices
        merged_buy = merged_buy_full[merged_buy_full.id.isin(common_sample_indices_set)].copy()
        merged_sell = merged_sell_full[merged_sell_full.id.isin(common_sample_indices_set)].copy()

        # Add suffix to id to distinguish buy/sell
        merged_buy_prefixed = merged_buy.copy()
        merged_buy_prefixed['id'] = merged_buy_prefixed['id'].astype(str) + '_buy'
        merged_sell_prefixed = merged_sell.copy()
        merged_sell_prefixed['id'] = merged_sell_prefixed['id'].astype(str) + '_sell'

        # Combine buy and sell
        combined_df = pd.concat([merged_buy_prefixed, merged_sell_prefixed], ignore_index=True)

        # --- Compute returns ---
        all_returns = []
        buy_returns = []
        sell_returns = []

        for _, row in combined_df.iterrows():
            data_array = np.array(row['merged_data'])
            returns = data_array - data_array[0]
            if row['id'].endswith('_buy'):
                buy_returns.append(returns)
                all_returns.append(returns)
            elif row['id'].endswith('_sell'):
                sell_returns_inverted = returns * -1.0
                sell_returns.append(sell_returns_inverted)
                all_returns.append(sell_returns_inverted)

        buy_returns = np.array(buy_returns)
        sell_returns = np.array(sell_returns)
        all_returns = np.array(all_returns)

        avg_all_returns = np.mean(all_returns, axis=0)
        std_all_returns = np.std(all_returns, axis=0)
        avg_buy_returns = np.mean(buy_returns, axis=0)
        std_buy_returns = np.std(buy_returns, axis=0)
        avg_sell_returns = np.mean(sell_returns, axis=0)
        std_sell_returns = np.std(sell_returns, axis=0)

        all_returns_stats[vol_key][approach] = {
            "avg_all": avg_all_returns,
            "std_all": std_all_returns,
            "avg_buy": avg_buy_returns,
            "std_buy": std_buy_returns,
            "avg_sell": avg_sell_returns,
            "std_sell": std_sell_returns,
            "n_all": len(all_returns),
            "n_buy": len(buy_returns),
            "n_sell": len(sell_returns)
        }

        print(f"\n[{vol_cfg['label']}][{approach_labels[approach]}] buy: {len(buy_returns)}, sell: {len(sell_returns)}, total: {len(all_returns)}")

# --- Plot: Compare all approaches for all volumes ---

import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

# For legend grouping
legendgroup_map = {
    "hist": "Hist",
    "heur": "Heur",
    "gen": "Gen"
}

# Use the time axis from the first available approach
for vol_key, vol_cfg in volume_experiments.items():
    for approach in ["hist", "heur", "gen"]:
        stats = all_returns_stats.get(vol_key, {}).get(approach, None)
        if stats is None:
            continue
        x_axis = list(range(len(stats["avg_all"])))
        color = vol_cfg["color"]
        dash = approach_linestyle[approach]
        label = f"{vol_cfg['label']} {approach_labels[approach]}"
        legendgroup = legendgroup_map[approach]
        # Plot avg_all
        fig.add_trace(go.Scatter(
            x=x_axis,
            y=stats["avg_all"],
            mode='lines',
            name=label + f" (N={stats['n_all']})",
            line=dict(color=color, width=2, dash=dash),
            legendgroup=legendgroup,
        ))
        # Optionally, plot buy/sell separately as thinner lines:
        # fig.add_trace(go.Scatter(
        #     x=x_axis,
        #     y=stats["avg_buy"],
        #     mode='lines',
        #     name=label + " Buy",
        #     line=dict(color=color, width=1, dash=dash),
        #     legendgroup=legendgroup,
        #     showlegend=False
        # ))
        # fig.add_trace(go.Scatter(
        #     x=x_axis,
        #     y=stats["avg_sell"],
        #     mode='lines',
        #     name=label + " Sell*-1",
        #     line=dict(color=color, width=1, dash=dash),
        #     legendgroup=legendgroup,
        #     showlegend=False
        # ))

fig.update_layout(
    title='Average Returns Evolution: All (Buy + Sell*-1) for 3 Approaches (hist/heur/gen) and 3 Volumes',
    xaxis_title='Time Steps',
    yaxis_title='Average Returns (Price - Initial Price)',
    width=1200,
    height=700,
    legend=dict(x=0.01, y=0.99, font=dict(size=13)),
    template='plotly_white'
)

fig.show()

print("\nPlotted average returns for all approaches (hist/heur/gen) and all volumes:")
for vol_key, vol_cfg in volume_experiments.items():
    for approach in ["hist", "heur", "gen"]:
        stats = all_returns_stats.get(vol_key, {}).get(approach, None)
        if stats is not None:
            print(f"- {vol_cfg['label']} {approach_labels[approach]}: All={stats['n_all']}, Buy={stats['n_buy']}, Sell={stats['n_sell']}")
print("  (Sell returns are multiplied by -1 for comparison)")
print("  Color: volume, Line style: approach (solid=hist, dash=heur, dot=gen)")



[V=485][Hist] buy: 112, sell: 112, total: 224

[V=485][Heur] buy: 112, sell: 112, total: 224

[V=485][Gen] buy: 81, sell: 81, total: 162

[V=300][Hist] buy: 112, sell: 112, total: 224

[V=300][Heur] buy: 112, sell: 112, total: 224

[V=300][Gen] buy: 79, sell: 79, total: 158

[V=75][Hist] buy: 112, sell: 112, total: 224

[V=75][Heur] buy: 112, sell: 112, total: 224

[V=75][Gen] buy: 82, sell: 82, total: 164



Plotted average returns for all approaches (hist/heur/gen) and all volumes:
- V=485 Hist: All=224, Buy=112, Sell=112
- V=485 Heur: All=224, Buy=112, Sell=112
- V=485 Gen: All=162, Buy=81, Sell=81
- V=300 Hist: All=224, Buy=112, Sell=112
- V=300 Heur: All=224, Buy=112, Sell=112
- V=300 Gen: All=158, Buy=79, Sell=79
- V=75 Hist: All=224, Buy=112, Sell=112
- V=75 Heur: All=224, Buy=112, Sell=112
- V=75 Gen: All=164, Buy=82, Sell=82
  (Sell returns are multiplied by -1 for comparison)
  Color: volume, Line style: approach (solid=hist, dash=heur, dot=gen)


# other plots - DIDN'T REFINE BELOW

In [4]:
all_series = all_returns

In [5]:
# # ===== unfiltered =====
# with open(f'/app/data_saved/{buy_exp_name}/filtered_merged.pkl', 'rb') as f:
#     merged_buy_full = pickle.load(f)

# with open(f'/app/data_saved/{sell_exp_name}/filtered_merged.pkl', 'rb') as f:
#     merged_sell_full = pickle.load(f)

# # Load b_dict files
# with open(f'/app/data_saved/{buy_exp_name}/filtered_b_dict.pkl', 'rb') as f:
#     b_dict_buy_full = pickle.load(f)

# with open(f'/app/data_saved/{sell_exp_name}/filtered_b_dict.pkl', 'rb') as f:
#     b_dict_sell_full = pickle.load(f)

# # Load m_dict files
# with open(f'/app/data_saved/{buy_exp_name}/filtered_m_dict.pkl', 'rb') as f:
#     m_dict_buy_full = pickle.load(f)

# with open(f'/app/data_saved/{sell_exp_name}/filtered_m_dict.pkl', 'rb') as f:
#     m_dict_sell_full = pickle.load(f)

# print(f"Buy dataset shape before filtration: {merged_buy_full.shape}")
# print(f"Sell dataset shape before filtration: {merged_sell_full.shape}")
# print(f"b_dict_buy keys before filtration: {len(b_dict_buy_full)}")
# print(f"b_dict_sell keys before filtration: {len(b_dict_sell_full)}")
# print(f"m_dict_buy keys before filtration: {len(m_dict_buy_full)}")
# print(f"m_dict_sell keys before filtration: {len(m_dict_sell_full)}")

# # Filter datasets to only include common sample indices
# merged_buy = merged_buy_full[merged_buy_full.id.isin(common_sample_indices_set)].copy()
# merged_sell = merged_sell_full[merged_sell_full.id.isin(common_sample_indices_set)].copy()

# # Filter b_dict and m_dict to only include common sample indices
# b_dict_buy = {k: v for k, v in b_dict_buy_full.items() if k in common_sample_indices_set}
# b_dict_sell = {k: v for k, v in b_dict_sell_full.items() if k in common_sample_indices_set}
# m_dict_buy = {k: v for k, v in m_dict_buy_full.items() if k in common_sample_indices_set}
# m_dict_sell = {k: v for k, v in m_dict_sell_full.items() if k in common_sample_indices_set}

# print(f"Filtered buy dataset shape: {merged_buy.shape}")
# print(f"Filtered sell dataset shape: {merged_sell.shape}")
# print(f"Filtered b_dict_buy keys: {len(b_dict_buy)}")
# print(f"Filtered b_dict_sell keys: {len(b_dict_sell)}")
# print(f"Filtered m_dict_buy keys: {len(m_dict_buy)}")
# print(f"Filtered m_dict_sell keys: {len(m_dict_sell)}")

In [6]:
# ===== specific filtered =====

# Load b_dict_buy.pkl
with open('/app/data_saved/exp_147_20251005_220723_gen_buy_75_b0_b7/filtered_b_dict.pkl', 'rb') as f:
    b_dict_buy = pickle.load(f)

# Load b_dict_sell.pkl
with open('/app/data_saved/exp_148_20251006_023201_gen_sell_75_b0_b7/filtered_b_dict.pkl', 'rb') as f:
    b_dict_sell = pickle.load(f)

# Load m_dict_buy.pkl
with open('/app/data_saved/exp_147_20251005_220723_gen_buy_75_b0_b7/filtered_m_dict.pkl', 'rb') as f:
    m_dict_buy = pickle.load(f)

# Load m_dict_sell.pkl
with open('/app/data_saved/exp_148_20251006_023201_gen_sell_75_b0_b7/filtered_m_dict.pkl', 'rb') as f:
    m_dict_sell = pickle.load(f)



# Merge m_dicts and b_dicts with _buy and _sell suffixes
m_dict_combined = {}
b_dict_combined = {}

# Add buy dictionaries with _buy suffix
for key, value in m_dict_buy.items():
    m_dict_combined[key] = value

for key, value in b_dict_buy.items():
    b_dict_combined[key] = value

# Add sell dictionaries with _sell suffix (add 1000000 to keys)
for key, value in m_dict_sell.items():
    m_dict_combined[key + 1000000] = value

for key, value in b_dict_sell.items():
    b_dict_combined[key + 1000000] = value

print(f"Combined m_dict keys: {len(m_dict_combined)}")
print(f"Combined b_dict keys: {len(b_dict_combined)}")

# Load sample_day_map and create combined version for buy and sell orders
# sample_day_map = pd.read_csv('/app/sample_day_map_1024.csv')

# Create a copy for sell orders with sample_id + 1000000
sample_day_map_sell = sample_day_map.copy()
sample_day_map_sell['sample_id'] = sample_day_map_sell['sample_id'] + 1000000

# Combine buy and sell mappings
sample_day_map_combined = pd.concat([sample_day_map, sample_day_map_sell], ignore_index=True)

print(f"Original sample_day_map rows: {len(sample_day_map)}")
print(f"Combined sample_day_map rows: {len(sample_day_map_combined)}")
print(f"Buy sample_ids range: {sample_day_map['sample_id'].min()} - {sample_day_map['sample_id'].max()}")
print(f"Sell sample_ids range: {sample_day_map_sell['sample_id'].min()} - {sample_day_map_sell['sample_id'].max()}")

sample_day_map = sample_day_map_combined

Combined m_dict keys: 178
Combined b_dict keys: 178
Original sample_day_map rows: 112
Combined sample_day_map rows: 224
Buy sample_ids range: 37 - 6177
Sell sample_ids range: 1000037 - 1006177


# Market impact graph

In [7]:
from IPython.display import display
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import ipywidgets as widgets


def calculate_impact(messages, valid_insertions, reference_price):
    """
    Calculate market impact for each insertion.
    
    Parameters
    ----------
    messages : np.array
        Message array with columns [EVENT_TYPE, DIRECTION, PRICE, REL, SIZE, ...]
    valid_insertions : list
        List of insertion indices
    reference_price : float
        Reference price at first insertion
        
    Returns
    -------
    impact : np.array
        Absolute impact for each insertion
    vwap_series : np.array
        VWAP series for each insertion
    Q_cum : np.array
        Cumulative quantity for each insertion
    log_imp : np.array
        Log of impact values for plotting
    """
    PRICE_COL = 3
    SIZE_COL = 5
    insert_sizes = messages[valid_insertions, SIZE_COL].astype(float)
    insert_prices = messages[valid_insertions, PRICE_COL].astype(float)
    Q_cum = np.cumsum(insert_sizes)
    notional = np.cumsum(insert_sizes * insert_prices)

    vwap_series = notional / np.maximum(Q_cum, 1e-12)
    # vwap_series = insert_prices
    
    impact = np.abs(vwap_series - reference_price) / reference_price
    
    # Calculate log impact for plotting (y-axis)
    eps = 1e-12
    log_imp = np.log(np.maximum(impact, eps))
    
    return impact, vwap_series, Q_cum, log_imp


def calculate_market_volume(messages, hist_steps, valid_insertions, execution_sum):
    """
    Calculate market execution volume V_exp for each insertion and return x-axis values for plotting.
    
    Parameters
    ----------
    messages : np.array
        Message array with columns [EVENT_TYPE, DIRECTION, PRICE, REL, SIZE, ...]
    hist_steps : int
        Starting index for market volume calculation
    valid_insertions : list
        List of insertion indices
    execution_sum : float
        Total execution sum for the day
        
    Returns
    -------
    V_exp : np.array
        Market volume from hist_steps to (idx-1) for each insertion
    log_qv : np.array
        Log of Q/V_exp ratio for plotting (x-axis)
    """
    EVENT_TYPE_COL = 1
    SIZE_COL = 5
    
    evt_types = messages[:, EVENT_TYPE_COL].astype(int)
    exec_sizes = np.where(evt_types == 4, messages[:, SIZE_COL].astype(float), 0.0)
    cum_exec_vol = np.cumsum(exec_sizes)
    
    V_exp = np.array([float(cum_exec_vol[idx-1] - cum_exec_vol[hist_steps-1] if (idx-1) >= hist_steps else 0.0)
                      for idx in valid_insertions])

    # Use execution_sum instead of fixed value
    V_exp = np.full_like(V_exp, execution_sum)
    
    # Calculate cumulative quantity for Q/V_exp ratio
    insert_sizes = messages[valid_insertions, SIZE_COL].astype(float)
    Q_cum = np.cumsum(insert_sizes)
    
    # Calculate log(Q/V_exp) for plotting (x-axis)
    eps = 1e-12
    rel_size = Q_cum / np.maximum(V_exp, eps)
    log_qv = np.log(np.maximum(rel_size, eps))

    return V_exp, log_qv


def calculate_book_volumes(book_array):
    """
    Calculate total bid and ask volumes from book data.
    
    Parameters
    ----------
    book_array : np.array
        Book array with L2 data
        
    Returns
    -------
    bid_volumes : np.array
        Total bid volume for each time step
    ask_volumes : np.array
        Total ask volume for each time step
    """
    # Assuming book data structure: first half is ask side, second half is bid side
    # Adjust slice according to your actual book layout
    T = book_array.shape[0]
    # l2_slice = slice(20, 800)  # adjust to your layout
    l2_slice = slice(0, 1000)  # adjust to your layout
    book_data = book_array[:, l2_slice]
    
    # Split into ask and bid sides (assuming symmetric layout)
    mid_idx = book_data.shape[1] // 2
    ask_side = book_data[:, :mid_idx]  # first half (negative levels)
    bid_side = book_data[:, mid_idx:]  # second half (positive levels)
    
    # Calculate total volumes (sum of absolute values for each side)
    ask_volumes = np.sum(np.abs(ask_side), axis=1)
    bid_volumes = np.sum(np.abs(bid_side), axis=1)
    
    return bid_volumes, ask_volumes


def interactive_market_impact_plot(
    b_seq_inp,            # dict[id] -> np.array(T, ...), where col 0 = Δmid per step
    msg_seq_raw,          # dict[id] -> np.array(T, num_fields)
    all_series,           # unused for mid now
    x,                    # time axis (len T)
    hist_steps=550,
    gen_block=50,
    num_insertions=20,
    beta_theory=0.5,      # theoretical slope
    tick_size=100,        # tick size for price conversion
):
    """
    • Q accumulates ONLY our insertions (from zero).
    • V_exp: cumulative market executions (event_type==4) from index hist_steps to (idx-1).
    • Impact = |VWAP_inserted - reference_price|.
    • Absolute mid reconstructed from cumulative Δmid (book[:,0]), anchored to ref price at first insertion.
    • In log–log panel: raw log values without normalization.
      - Allowed points (used for fit): colored
      - Zero-impact points: grey, excluded from fit
    • Theoretical line: passes through fixed intercept with slope beta_theory.
    • Prices converted from ticks to dollars using tick_size.
    • 4th graph: Evolution of total available volume on ask and bid sides.
    """

    # Accept DataFrame inputs -> dict[int] -> np.array
    if isinstance(b_seq_inp, pd.DataFrame):
        b_seq_inp = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
    if isinstance(msg_seq_raw, pd.DataFrame):
        msg_seq_raw = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}

    # --- UI ---
    id_dd       = widgets.Dropdown(options=sorted(b_seq_inp.keys()), description="Sample ID:")
    time_slider = widgets.IntSlider(min=1, max=1, step=1, description="t:")
    btn_prev    = widgets.Button(description="←")
    btn_next    = widgets.Button(description="→")
    msg_box     = widgets.HTML()
    coeff_box   = widgets.HTML()

    if 79 in b_seq_inp:
        id_dd.value = 79

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=["Book state (ΔL2)", "Mid (absolute)", "Market Impact (log–log)", "Book Volumes (Ask/Bid)"]
    )
    fig.update_layout(width=1400, height=1040, showlegend=True, template='plotly_white')
    fig_widget = go.FigureWidget(fig)

    # raw message column indices (adjust if needed)
    EVENT_TYPE_COL = 1
    DIRECTION_COL  = 2
    PRICE_COL      = 3   # absolute price in ticks
    REL_COL        = 4
    SIZE_COL       = 5

    def update_slider_range(*_):
        arr = b_seq_inp[id_dd.value]
        time_slider.min = 1
        time_slider.max = arr.shape[0] - 1
        time_slider.value = min(551, time_slider.max)

    def update_plot(*_):
        sample_id = id_dd.value
        t = time_slider.value

        book_array = b_seq_inp[sample_id]      # [T, ...]
        messages   = msg_seq_raw[sample_id]    # [T, num_fields]
        T = len(messages)

        # Get day data for this sample from sample_day_map
        sample_row = sample_day_map[sample_day_map['sample_id'] == sample_id]
        if sample_row.empty:
            coeff_box.value = f"<b>Sample {sample_id} not found in sample_day_map.</b>"
            with fig_widget.batch_update():
                fig_widget.data = []
                fig_widget.layout.shapes = []
            return
        
        highest_price = sample_row.iloc[0]['highest_price']
        lowest_price = sample_row.iloc[0]['lowest_price']
        execution_sum = sample_row.iloc[0]['execution_sum']
        
        # (1) Book viz (ΔL2 slices)
        # l2_slice = slice(240, 263)  # adjust to your layout
        # l2_slice = slice(20, 800)  # adjust to your layout
        l2_slice = slice(0, 1000)  # adjust to your layout
        book_prev = book_array[t-1, l2_slice]
        book_now  = book_array[t,   l2_slice]
        book_diff = np.abs(book_now) - np.abs(book_prev)
        x_lvls    = np.arange(len(book_prev)) - len(book_prev)//2
        book_cols = ['orange' if abs(d) < 1e-8 else ('red' if d > 0 else 'blue') for d in book_diff]

        # (2) Insertion positions
        insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
        valid_insertions = [pos for pos in insertion_positions if pos < T]
        if not valid_insertions:
            coeff_box.value = "<b>No valid insertions.</b>"
            with fig_widget.batch_update():
                fig_widget.data = []
                fig_widget.layout.shapes = []
            return

        # (3) Reference absolute price at first insertion (convert from ticks to dollars)
        ref_idx = valid_insertions[0]
        reference_price = float(messages[ref_idx, PRICE_COL]) / tick_size

        # (4) Absolute mid reconstruction (for middle chart and eta calculation)
        delta_mid = book_array[:, 0].astype(float) / tick_size  # Δmid_t in dollars
        cum_delta_mid   = np.cumsum(delta_mid)
        mid_abs   = reference_price + (cum_delta_mid - cum_delta_mid[ref_idx])
        mid_series_abs  = mid_abs

        # (5) Calculate impact and market volume using helper functions - get x and y for plotting
        # Convert prices to dollars in the helper functions
        messages_dollars = messages.copy().astype(float)
        messages_dollars[:, PRICE_COL] /= tick_size
        
        impact, vwap_series, Q_cum, log_imp = calculate_impact(messages_dollars, valid_insertions, reference_price)
        V_exp, log_qv = calculate_market_volume(messages_dollars, hist_steps, valid_insertions, execution_sum)

        # (6) Calculate book volumes for 4th graph
        bid_volumes, ask_volumes = calculate_book_volumes(book_array)

        # (7) Calculate eta for display using highest_price and lowest_price from day data
        H = highest_price / tick_size  # Convert to dollars
        L = lowest_price / tick_size   # Convert to dollars
        print(f"DEBUG eta calculation: H={H:.10f}, L={L:.10f}")
        
        if H > L:
            ln_hl = np.log(H / L)
            print(f"DEBUG eta calculation: ln(H/L)={ln_hl:.10f}")
            eta_day = ln_hl / 0.8325546
            print(f"DEBUG eta calculation: eta_day = {ln_hl:.10f} / 0.8325546 = {eta_day:.10f}")
            print(f"DEBUG ln(eta) calculation: ln(eta_day) = {np.log(eta_day):.10f}")
        else:
            eta_day = 1e-12
            print(f"DEBUG eta calculation: H <= L, using eta_day = {eta_day:.10f}")

        # (8) Determine which points to use for fitting
        tol = 1e-12
        mask_zero = impact <= tol
        mask_pos  = ~mask_zero

        # (9) Fit on allowed points
        used_x_raw = log_qv[mask_pos]
        used_y_raw = log_imp[mask_pos]

        if used_x_raw.size >= 2:
            # Fit in raw log space
            # Calculate fixed intercept based on high/low of midprices
            fixed_intercept = np.log(eta_day)
            
            # Fit regression with fixed intercept
            # y = fixed_intercept + beta * x, so we solve for beta using: beta = mean((y - fixed_intercept) / x)
            adjusted_y = used_y_raw - fixed_intercept
            beta_hat = float(np.mean(adjusted_y / used_x_raw))
            alpha_hat = fixed_intercept

            # Calculate min/max and deltas for allowed points only
            x_min, x_max = float(used_x_raw.min()), float(used_x_raw.max())
            y_min, y_max = float(used_y_raw.min()), float(used_y_raw.max())
            x_delta = x_max - x_min
            y_delta = y_max - y_min

            # Theoretical line in raw log space: y = fixed_intercept + beta_theory * x
            n_used = used_x_raw.shape[0]; n_total = len(log_qv)
            coeff_box.value = f"""
            <div style="padding-left:20px; font-family:monospace">
                <h4>Market Impact (log–log; allowed points only) - Sample {sample_id}</h4>
                <p><b>Fit:</b> log(Impact) = <b>{alpha_hat:.10f}</b> + <b>{beta_hat:.10f}</b> · log(Q/V_exp)</p>
                <p><b>Theory:</b> log(Impact) = {fixed_intercept:.10f} + {beta_theory:.2f} · log(Q/V_exp)</p>
                <p><b>Eta:</b> {eta_day:.10f}</p>
                <p><b>H:</b> ${H:.2f}, <b>L:</b> ${L:.2f}, <b>Execution Sum:</b> {execution_sum:.0f}</p>
                <p>Used points: {n_used} / {n_total}</p>
                <p><b>X range:</b> [{x_min:.10f}, {x_max:.10f}] (Δ={x_delta:.10f})</p>
                <p><b>Y range:</b> [{y_min:.10f}, {y_max:.10f}] (Δ={y_delta:.10f})</p>
            </div>
            """

            # Line ranges for plotting
            xspan = np.linspace(float(used_x_raw.min()), float(used_x_raw.max()), 100)
            fit_y = alpha_hat + beta_hat * xspan
            th_y  = fixed_intercept + beta_theory * xspan

        else:
            xspan = np.array([]); fit_y = np.array([]); th_y = np.array([])
            coeff_box.value = f"""
            <div style="padding-left:20px; font-family:monospace">
                <h4>Market Impact (log–log) - Sample {sample_id}</h4>
                <p><b>Fit:</b> not enough non-zero impact points.</p>
                <p><b>Eta:</b> {eta_day:.10f}</p>
                <p><b>H:</b> ${H:.2f}, <b>L:</b> ${L:.2f}, <b>Execution Sum:</b> {execution_sum:.0f}</p>
            </div>
            """

        # Labels 1..N
        insert_labels = [str(i) for i, _ in enumerate(valid_insertions, start=1)]

        # (10) Draw
        with fig_widget.batch_update():
            fig_widget.data = []
            fig_widget.layout.shapes = []

            # (1) L2 book bars
            fig_widget.add_bar(x=x_lvls, y=book_prev, row=1, col=1, marker_color='orange', name="Prev L2")
            fig_widget.add_bar(x=x_lvls, y=book_now,  row=1, col=1, marker_color=book_cols, name="Now L2")

            # (2) mid (absolute)
            fig_widget.add_scatter(x=np.arange(T), y=mid_series_abs, mode='lines',
                                   row=1, col=2, line=dict(width=1), name="Mid (abs)")
            fig_widget.add_shape(type="line", x0=t, x1=t,
                                 y0=float(np.nanmin(mid_series_abs)),
                                 y1=float(np.nanmax(mid_series_abs)),
                                 line=dict(color="green", width=2), xref="x2", yref="y2")

            # insertion markers on mid
            insert_dots_x = [p for p in valid_insertions if p < len(mid_series_abs)]
            insert_dots_y = [mid_series_abs[p] for p in insert_dots_x]
            fig_widget.add_scatter(
                x=insert_dots_x, y=insert_dots_y, mode='markers',
                marker=dict(size=7, symbol='circle'),
                text=insert_labels[:len(insert_dots_x)],
                hovertemplate="Insertion %{text}<extra></extra>",
                row=1, col=2, name="Insert marks"
            )

            # (3) raw log–log points & lines
            if xspan.size > 0:
                # allowed (impact>0)
                fig_widget.add_scatter(
                    x=log_qv[mask_pos], y=log_imp[mask_pos], mode='markers',
                    marker=dict(size=8),
                    text=[lbl for lbl, m in zip(insert_labels, mask_pos) if m],
                    hovertemplate="Insertion %{text}<extra></extra>",
                    row=2, col=1, name="Points (allowed)"
                )
                # zero-impact -> grey
                if np.any(~mask_pos):
                    fig_widget.add_scatter(
                        x=log_qv[~mask_pos], y=log_imp[~mask_pos], mode='markers',
                        marker=dict(size=8, color='grey'),
                        text=[lbl for lbl, m in zip(insert_labels, ~mask_pos) if m],
                        hovertemplate="Insertion %{text} (zero-impact)<extra></extra>",
                        row=2, col=1, name="Zero-impact (grey)"
                    )
                # fitted & theoretical lines
                fig_widget.add_scatter(x=xspan, y=fit_y, mode='lines',
                                       line=dict(dash='dot', width=2), row=2, col=1, name="Fit")
                fig_widget.add_scatter(x=xspan, y=th_y, mode='lines',
                                       line=dict(dash='dash', width=2), row=2, col=1, name=f"Theoretical (β={beta_theory:.2f})")

                fig_widget.update_xaxes(title="log(Q / V_exp)", row=2, col=1)
                fig_widget.update_yaxes(title="log(Impact)",   row=2, col=1)
            else:
                fig_widget.update_xaxes(title="log(Q / V_exp)", row=2, col=1)
                fig_widget.update_yaxes(title="log(Impact)",   row=2, col=1)

            # (4) Book volumes evolution
            fig_widget.add_scatter(x=np.arange(T), y=bid_volumes, mode='lines',
                                   line=dict(color='blue', width=1), row=2, col=2, name="Bid Volume")
            fig_widget.add_scatter(x=np.arange(T), y=ask_volumes, mode='lines',
                                   line=dict(color='red', width=1), row=2, col=2, name="Ask Volume")
            
            # Current time vertical line for volumes
            fig_widget.add_shape(type="line", x0=t, x1=t,
                                 y0=float(min(np.nanmin(bid_volumes), np.nanmin(ask_volumes))),
                                 y1=float(max(np.nanmax(bid_volumes), np.nanmax(ask_volumes))),
                                 line=dict(color="green", width=2), xref="x4", yref="y4")

            # insertion markers on volumes
            insert_dots_x_vol = [p for p in valid_insertions if p < len(bid_volumes)]
            insert_dots_y_bid = [bid_volumes[p] for p in insert_dots_x_vol]
            insert_dots_y_ask = [ask_volumes[p] for p in insert_dots_x_vol]
            
            fig_widget.add_scatter(
                x=insert_dots_x_vol, y=insert_dots_y_bid, mode='markers',
                marker=dict(size=7, symbol='circle', color='blue'),
                text=insert_labels[:len(insert_dots_x_vol)],
                hovertemplate="Insertion %{text} - Bid<extra></extra>",
                row=2, col=2, name="Insert marks (Ask)"
            )
            fig_widget.add_scatter(
                x=insert_dots_x_vol, y=insert_dots_y_ask, mode='markers',
                marker=dict(size=7, symbol='circle', color='red'),
                text=insert_labels[:len(insert_dots_x_vol)],
                hovertemplate="Insertion %{text} - Ask<extra></extra>",
                row=2, col=2, name="Insert marks (Bid)"
            )
            
            fig_widget.update_xaxes(title="Time", row=2, col=2)
            fig_widget.update_yaxes(title="Volume", row=2, col=2)

            # highlight current insertion
            if xspan.size > 0 and t in valid_insertions:
                i_sel = valid_insertions.index(t)
                x_sel = log_qv[i_sel]
                y_sel = log_imp[i_sel]
                fig_widget.add_scatter(
                    x=[x_sel], y=[y_sel], mode='markers',
                    marker=dict(color='red', size=12),
                    text=[insert_labels[i_sel]],
                    hovertemplate="Insertion %{text}<extra></extra>",
                    row=2, col=1, name="Current"
                )

        # raw message info @ t (display prices in dollars)
        m = messages[t].astype(int)
        price_dollars = m[PRICE_COL] / tick_size
        event_map = {1: "Limit", 2: "PartialCancel", 3: "Delete", 4: "Execution"}
        direction_map = {1: "Buy", 0: "Sell"}
        msg_box.value = (
            f"<b>{event_map.get(m[EVENT_TYPE_COL], '?')} • {direction_map.get(m[DIRECTION_COL], '?')} "
            f"• abs=${price_dollars:.2f} • rel={m[REL_COL]} • size={m[SIZE_COL]}</b><br>raw: {m.tolist()}"
        )

    def on_prev(_):
        if time_slider.value > time_slider.min:
            time_slider.value -= 1

    def on_next(_):
        if time_slider.value < time_slider.max:
            time_slider.value += 1

    id_dd.observe(lambda _: update_slider_range(), names='value')
    time_slider.observe(lambda _: update_plot(), names='value')
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)

    update_slider_range()
    update_plot()

    display(widgets.HBox([id_dd, btn_prev, btn_next, time_slider]))
    display(widgets.HBox([fig_widget, coeff_box]))
    display(msg_box)


In [8]:
# experiment_name = 'exp_89_20250828_194553_gen_sell_1024'  #       exp_85_20250823_020427_1024
# # filtration_name = 'samples_after_book_filtration_buy_1024_446'

CONFIG_PATH = f"/app/data_saved/exp_147_20251005_220723_gen_buy_75_b0_b7/used_config.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)
num_insertions      = config["num_insertions"]
num_coolings        = config["num_coolings"]
midprice_step_size  = config["midprice_step_size"]
hist_msgs           = config["n_messages"]
n_gen_msgs          = config["n_gen_msgs"]

hist_steps = hist_msgs // midprice_step_size       # 500
gen_steps = n_gen_msgs // midprice_step_size     # 50
gen_block = gen_steps + 1                        # 51

# x is just numbers from 1 to max len (all_series)
x = list(range(1, all_series.shape[1] + 1))

In [9]:
# all_betas = interactive_market_impact_plot(b_dict_combined, m_dict_combined, all_series, x, hist_steps, gen_block, num_insertions)

In [10]:
import numpy as np

# ---------- small utils ----------
def _as_float(a):
    return np.asarray(a, dtype=float)

def _mask_xy(x, y):
    x = _as_float(x); y = _as_float(y)
    m = np.isfinite(x) & np.isfinite(y) & (x != 0.0)
    return x[m], y[m], m

def _beta_wls_fixed_internal(x, y_adj, w):
    # Weighted regression of y_adj on x with intercept fixed at 0
    x = _as_float(x); y_adj = _as_float(y_adj); w = _as_float(w)
    m = np.isfinite(x) & np.isfinite(y_adj) & np.isfinite(w) & (x != 0) & (w > 0)
    if m.sum() < 2: return np.nan
    xw = x[m] * np.sqrt(w[m]); yw = y_adj[m] * np.sqrt(w[m])
    denom = np.dot(xw, xw)
    if denom <= 0: return np.nan
    return float(np.dot(xw, yw) / denom)

# ---------- estimators (fixed intercept y = alpha + beta * x) ----------
def _beta_ols_fixed(x, y, alpha):
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    denom = np.dot(x, x)
    if denom <= 0: return np.nan
    return float(np.dot(x, y_adj) / denom)

def _beta_huber_fixed(x, y, alpha, c=1.345, max_iter=50, tol=1e-8):
    # Huber M via IRLS (defaults chosen for ~95% Gaussian efficiency)
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    beta = _beta_ols_fixed(x, y, alpha)
    if not np.isfinite(beta): beta = 0.0
    for _ in range(max_iter):
        r = y_adj - beta * x
        med = np.median(r)
        sigma = 1.4826 * np.median(np.abs(r - med)) or (np.std(r) + 1e-12)
        u = r / (sigma + 1e-12)
        w = np.ones_like(u)
        big = np.abs(u) > c
        w[big] = (c / (np.abs(u[big]) + 1e-12))
        beta_new = _beta_wls_fixed_internal(x, y_adj, w)
        if not np.isfinite(beta_new): break
        if abs(beta_new - beta) <= tol * (1.0 + abs(beta)):
            beta = beta_new; break
        beta = beta_new
    return float(beta)

def _beta_lad_fixed(x, y, alpha, iters=100, eps=1e-8):
    # LAD (L1) via IRLS: w_i = 1/max(|r_i|, eps)
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    beta = _beta_ols_fixed(x, y, alpha)
    if not np.isfinite(beta): beta = 0.0
    for _ in range(iters):
        r = y_adj - beta * x
        w = 1.0 / np.maximum(np.abs(r), eps)
        beta_new = _beta_wls_fixed_internal(x, y_adj, w)
        if not np.isfinite(beta_new): break
        if abs(beta_new - beta) <= 1e-7 * (1.0 + abs(beta)):
            beta = beta_new; break
        beta = beta_new
    return float(beta)

def _beta_ratio_median(x, y, alpha, x_floor=1e-6):
    # Median of ratios with |x| floor (avoid blow-ups near 0)
    x = _as_float(x); y_adj = _as_float(y) - _as_float(alpha)
    m = np.isfinite(x) & np.isfinite(y_adj) & (np.abs(x) >= x_floor)
    r = y_adj[m] / x[m]
    if r.size == 0: return np.nan
    return float(np.median(np.sort(r)))

def _beta_ratio_trim(x, y, alpha, x_floor=1e-6, trim=0.10):
    # Trimmed-mean of ratios (default 10% each tail) with |x| floor
    x = _as_float(x); y_adj = _as_float(y) - _as_float(alpha)
    m = np.isfinite(x) & np.isfinite(y_adj) & (np.abs(x) >= x_floor)
    r = np.sort(y_adj[m] / x[m])
    if r.size == 0: return np.nan
    k = int(trim * r.size)
    r = r[k: r.size - k] if r.size - 2*k > 0 else r
    return float(np.mean(r))

def _beta_deming_fixed(x, y, alpha, lambda_yx=1.0):
    # Orthogonal regression with fixed intercept (through origin on y_adj)
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    s_xx = np.dot(x, x) / x.size
    s_yy = np.dot(y_adj, y_adj) / x.size
    s_xy = np.dot(x, y_adj) / x.size
    if s_xy == 0.0: return np.nan
    A = s_yy - lambda_yx * s_xx
    B = 2.0 * s_xy
    disc = A*A + (B*B) * lambda_yx
    beta = (A + np.sqrt(disc)) / B
    return float(beta)

# ---------- single public entry ----------
def beta_fit(x, y, alpha, method="ols"):
    """
    Estimate beta in y = alpha + beta * x with a fixed intercept.

    Parameters
    ----------
    x, y : array-like
    alpha : float or array-like (broadcastable)
    method : {'ols','huber','lad','ratio-median','ratio-trim','deming'}

    Returns
    -------
    beta : float
    """
    m = method.lower()
    if m == "ols":
        return _beta_ols_fixed(x, y, alpha)
    elif m == "huber":
        return _beta_huber_fixed(x, y, alpha)           # c=1.345, 50 iters, tol=1e-8
    elif m == "lad":
        return _beta_lad_fixed(x, y, alpha)             # 100 iters, eps=1e-8
    elif m == "ratio-median":
        return _beta_ratio_median(x, y, alpha)          # x_floor=1e-6
    elif m == "ratio-trim":
        return _beta_ratio_trim(x, y, alpha)            # trim=10%, x_floor=1e-6
    elif m == "deming":
        return _beta_deming_fixed(x, y, alpha)          # lambda_yx=1.0
    else:
        raise ValueError(f"Unknown method '{method}'. Use one of: "
                         "ols, huber, lad, ratio-median, ratio-trim, deming.")

In [11]:
# --- Compute points_df and coeffs_df for beta evolution plot only ---

est_method = "lad"

def compute_points_and_coeffs(
    b_seq_inp,
    msg_seq_raw,
    hist_steps=550,
    gen_block=50,
    num_insertions=20,
    tick_size=100,
):
    """
    Compute points_df and coeffs_df for beta(a) evolution plot.
    """
    # Normalize inputs
    if isinstance(b_seq_inp, pd.DataFrame):
        b_dict_local = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
    else:
        b_dict_local = b_seq_inp
    if isinstance(msg_seq_raw, pd.DataFrame):
        m_dict_local = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}
    else:
        m_dict_local = msg_seq_raw

    EVENT_TYPE_COL = 1
    PRICE_COL      = 3
    SIZE_COL       = 5

    eps = 1e-12
    tol = 1e-12

    sample_ids = sorted(set(b_dict_local.keys()) & set(m_dict_local.keys()))
    col_names = [f"ins_{i}" for i in range(1, num_insertions + 1)]
    x_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
    y_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
    coeff_rows = []

    for sid in sample_ids:
        messages_ticks = m_dict_local[sid]
        book = b_dict_local[sid]
        T = len(messages_ticks)

        # insertion schedule
        insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
        valid_insertions = [pos for pos in insertion_positions if pos < T]
        if not valid_insertions:
            coeff_rows.append({"sample_id": sid, "alpha_hat": np.nan, "beta_hat": np.nan,
                               "n_used": 0, "n_total": 0})
            continue

        # Reference price at first insertion (ticks -> $)
        ref_idx = valid_insertions[0]
        reference_price = float(messages_ticks[ref_idx, PRICE_COL]) / tick_size

        # Get day info from sample_day_map if available, else fallback
        try:
            day_row = sample_day_map[sample_day_map['sample_id'] == sid]
            if not day_row.empty:
                H_ticks = float(day_row.iloc[0]['highest_price'])
                L_ticks = float(day_row.iloc[0]['lowest_price'])
                execution_sum = float(day_row.iloc[0]['execution_sum'])
            else:
                H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
                L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
                exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
                execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))
        except Exception:
            H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
            L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
            exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
            execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))

        # Convert to dollars for Parkinson eta
        H = float(H_ticks) / tick_size
        L = float(L_ticks) / tick_size
        if np.isfinite(H) and np.isfinite(L) and H > L and L > 0:
            eta_day = np.log(H / L) / 0.8325546
            alpha_fixed = float(np.log(max(eta_day, eps)))   # α = ln(η)
        else:
            eta_day = eps
            alpha_fixed = float(np.log(eta_day))

        # Convert messages to dollars for the helper functions
        messages_dollars = messages_ticks.astype(float).copy()
        messages_dollars[:, PRICE_COL] /= tick_size

        # --- use helper functions ---
        impact, vwap_series, Q_cum, log_imp = calculate_impact(
            messages_dollars, valid_insertions, reference_price
        )
        V_exp, log_qv = calculate_market_volume(
            messages_dollars, hist_steps, valid_insertions, execution_sum
        )

        # fill tables per insertion
        mask_zero = impact <= tol
        mask_pos  = ~mask_zero
        for j, _idx in enumerate(valid_insertions):
            col = f"ins_{j+1}"
            if mask_zero[j] or not np.isfinite(log_qv[j]) or not np.isfinite(log_imp[j]):
                x_df.loc[sid, col] = "ZERO"
                y_df.loc[sid, col] = "ZERO"
            else:
                x_df.loc[sid, col] = float(log_qv[j])
                y_df.loc[sid, col] = float(log_imp[j])

        # per-sample β with fixed intercept
        used_x = log_qv[mask_pos]
        used_y = log_imp[mask_pos]
        n_used = int(used_x.size)
        n_total = int(len(valid_insertions))
        if n_used >= 2 and np.all(np.isfinite(used_x)) and np.all(np.isfinite(used_y)):
            valid_mask = (used_x != 0) & np.isfinite(used_x) & np.isfinite(used_y)
            if np.sum(valid_mask) >= 2:
                beta_hat = beta_fit(used_x[valid_mask], used_y[valid_mask], alpha_fixed, method=est_method)
            else:
                beta_hat = np.nan
        else:
            beta_hat = np.nan

        coeff_rows.append({
            "sample_id": sid,
            "alpha_hat": alpha_fixed,
            "beta_hat": beta_hat,
            "n_used": n_used,
            "n_total": n_total,
        })

    coeffs_df = pd.DataFrame.from_records(coeff_rows).set_index("sample_id").sort_index()

    # Build points_df
    rows = []
    for sid in x_df.index:
        for j, col in enumerate(x_df.columns, start=1):
            xv = x_df.loc[sid, col]
            yv = y_df.loc[sid, col]
            if isinstance(xv, (int, float, np.floating)) and isinstance(yv, (int, float, np.floating)):
                if np.isfinite(xv) and np.isfinite(yv):
                    rows.append({"sample_id": sid, "insertion": j, "x": float(xv), "y": float(yv)})
    points_df = pd.DataFrame(rows)
    return points_df, coeffs_df

# --- Compute points and coeffs ---
points_df, coeffs_df = compute_points_and_coeffs(
    b_seq_inp=b_dict_combined,
    msg_seq_raw=m_dict_combined,
    hist_steps=hist_steps,
    gen_block=gen_block,
    num_insertions=num_insertions,
    tick_size=100,
)

# --- Beta(a) evolution plot only ---

import plotly.graph_objects as go
import numpy as np

method = "lad"
method_label = "LAD"
color = "#1f77b4"

fig = go.Figure()

max_insertions = points_df["insertion"].max() if not points_df.empty else 10
a_values = np.arange(1, max_insertions + 1)
betas_evo = np.full_like(a_values, np.nan, dtype=float)

alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

for idx, a in enumerate(a_values):
    mask = points_df["insertion"] >= a
    X = points_df.loc[mask, "x"].to_numpy()
    Y = points_df.loc[mask, "y"].to_numpy()
    if len(Y) >= 2:
        valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        if np.sum(valid_mask) >= 2:
            x_valid = X[valid_mask]
            y_valid = Y[valid_mask]
            try:
                beta = beta_fit(x_valid, y_valid, alpha_global, method=method)
                betas_evo[idx] = beta
            except Exception:
                betas_evo[idx] = np.nan

fig.add_trace(go.Scatter(
    x=a_values,
    y=betas_evo,
    mode='lines+markers',
    name=f'{method_label} regression',
    line=dict(color=color, width=2),
    marker=dict(size=6, color=color)
))

fig.add_trace(go.Scatter(
    x=[a_values[0], a_values[-1]],
    y=[0.5, 0.5],
    mode='lines',
    name='Theoretical β = 0.5',
    line=dict(color='black', dash='dash', width=2)
))

fig.update_layout(
    title=dict(
        text='Beta Evolution with LAD Regression',
        font=dict(size=16, family="Arial Black")
    ),
    xaxis=dict(
        title='a (insertion threshold)',
        title_font=dict(size=14),
        tickfont=dict(size=12)
    ),
    yaxis=dict(
        title='β (slope)',
        title_font=dict(size=14),
        tickfont=dict(size=12),
        range=[0.0, 1.0]
    ),
    template='plotly_white',
    width=800,
    height=800,
    legend=dict(
        font=dict(size=12),
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    showlegend=True
)

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

fig.show()


In [12]:
# === Beta(a) evolution for all volumes & approaches (buy+sell combined BEFORE beta) ===
# Requirements: you already have:
# - volume_experiments, approach_linestyle, approach_labels (as in your snippet)
# - functions: compute_points_and_coeffs, calculate_impact, calculate_market_volume, beta_fit
# - file /app/batches_equal_sample_day_map.csv
# - each experiment folder contains: filtered_b_dict.pkl, filtered_m_dict.pkl, used_config.yaml
#
# Notes:
# - Comments are in English only, as requested.
# - This cell is self-contained w.r.t. loading data and plotting.
# - It safely skips missing experiments and reports what was plotted.

import os
import pickle
import yaml
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Safety defaults in case mappings are not in scope ---
approach_linestyle = approach_linestyle if 'approach_linestyle' in globals() else {"hist": "dot", "heur": "dash", "gen": "solid"}
approach_labels    = approach_labels    if 'approach_labels'    in globals() else {"hist": "Hist", "heur": "Heur", "gen": "Gen"}

# --- Load base day map once; we'll clone & offset for sell each loop ---
BASE_DAY_MAP_PATH = '/app/batches_equal_sample_day_map.csv'
base_day_map = pd.read_csv(BASE_DAY_MAP_PATH)

# --- LAD by default; change to "ols" if needed ---
method = "lad"
method_label = "LAD" if method.lower() == "lad" else method.upper()

# --- Storage for programmatic access after plotting ---
beta_curves = {}   # keys: (vol_key, approach) -> dict(a_values, betas, alpha_global, n_samples, n_points)

fig = go.Figure()

# --- Iterate volumes & approaches ---
for vol_key, vol_cfg in volume_experiments.items():
    color = vol_cfg.get("color", None)

    for approach in ["hist", "heur", "gen"]:
        exp = vol_cfg["approaches"].get(approach)
        if exp is None:
            print(f"[SKIP] {vol_key}-{approach}: no config")
            continue

        buy_dir  = f'/app/data_saved/{exp["buy"]}'
        sell_dir = f'/app/data_saved/{exp["sell"]}'
        b_buy_pkl  = os.path.join(buy_dir,  'filtered_b_dict.pkl')
        b_sell_pkl = os.path.join(sell_dir, 'filtered_b_dict.pkl')
        m_buy_pkl  = os.path.join(buy_dir,  'filtered_m_dict.pkl')
        m_sell_pkl = os.path.join(sell_dir, 'filtered_m_dict.pkl')
        cfg_path   = os.path.join(buy_dir,  'used_config.yaml')  # read from buy side (assumed same as sell)

        # --- Load dicts ---
        try:
            with open(b_buy_pkl, 'rb')  as f: b_buy  = pickle.load(f)
            with open(b_sell_pkl, 'rb') as f: b_sell = pickle.load(f)
            with open(m_buy_pkl, 'rb')  as f: m_buy  = pickle.load(f)
            with open(m_sell_pkl, 'rb') as f: m_sell = pickle.load(f)
        except Exception as e:
            print(f"[SKIP] {vol_key}-{approach}: cannot load dicts -> {e}")
            continue

        # --- Combine buy+sell by offsetting sell sample_ids by +1_000_000 ---
        OFFSET = 1_000_000
        m_combined, b_combined = {}, {}
        # buy as-is
        for k, v in (m_buy.items() if isinstance(m_buy, dict) else {}):
            m_combined[int(k)] = v
        for k, v in (b_buy.items() if isinstance(b_buy, dict) else {}):
            b_combined[int(k)] = v
        # sell with offset
        for k, v in (m_sell.items() if isinstance(m_sell, dict) else {}):
            m_combined[int(k) + OFFSET] = v
        for k, v in (b_sell.items() if isinstance(b_sell, dict) else {}):
            b_combined[int(k) + OFFSET] = v

        # --- Read config for this experiment (hist/gen/heur) ---
        try:
            with open(cfg_path, 'r') as f:
                cfg = yaml.safe_load(f)
            num_insertions     = int(cfg["num_insertions"])
            midprice_step_size = int(cfg["midprice_step_size"])
            hist_msgs          = int(cfg["n_messages"])
            n_gen_msgs         = int(cfg["n_gen_msgs"])
            tick_size          = int(cfg.get("tick_size", 100))
            hist_steps = hist_msgs // midprice_step_size
            gen_steps  = n_gen_msgs // midprice_step_size
            gen_block  = gen_steps + 1
        except Exception as e:
            # Fallback to safe defaults if config missing
            print(f"[WARN] {vol_key}-{approach}: cannot parse used_config.yaml -> {e}")
            num_insertions = 10
            tick_size = 100
            hist_steps = 500
            gen_steps  = 50
            gen_block  = gen_steps + 1

        # --- Build combined sample_day_map (duplicate with +OFFSET for sell) ---
        sample_day_map_sell = base_day_map.copy()
        sample_day_map_sell["sample_id"] = sample_day_map_sell["sample_id"].astype(int) + OFFSET
        sample_day_map = pd.concat([base_day_map, sample_day_map_sell], ignore_index=True)

        # --- Compute points & per-sample coeffs (uses global sample_day_map) ---
        try:
            points_df, coeffs_df = compute_points_and_coeffs(
                b_seq_inp=b_combined,
                msg_seq_raw=m_combined,
                hist_steps=hist_steps,
                gen_block=gen_block,
                num_insertions=num_insertions,
                tick_size=tick_size,
            )
        except Exception as e:
            print(f"[SKIP] {vol_key}-{approach}: compute_points_and_coeffs failed -> {e}")
            continue

        if points_df.empty:
            print(f"[SKIP] {vol_key}-{approach}: no points")
            continue

        # --- Global alpha: mean per-sample alpha_hat ---
        try:
            alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True))
        except Exception:
            alpha_global = 0.0

        # --- Build beta(a) curve ---
        max_a_from_points = int(points_df["insertion"].max())
        a_values = np.arange(1, max(2, min(num_insertions, max_a_from_points)) + 1)  # at least up to 2
        betas_evo = np.full_like(a_values, np.nan, dtype=float)

        for idx, a in enumerate(a_values):
            mask = points_df["insertion"] >= a
            X = points_df.loc[mask, "x"].to_numpy()
            Y = points_df.loc[mask, "y"].to_numpy()
            valid = np.isfinite(X) & np.isfinite(Y) & (X != 0)
            if np.sum(valid) >= 2:
                try:
                    beta = beta_fit(X[valid], Y[valid], alpha_global, method=method)
                    betas_evo[idx] = float(beta)
                except Exception:
                    betas_evo[idx] = np.nan

        # --- Plot trace ---
        label = f"{vol_cfg['label']} {approach_labels[approach]}"
        dash  = approach_linestyle[approach]
        n_samples = int(points_df["sample_id"].nunique())
        fig.add_trace(go.Scatter(
            x=a_values,
            y=betas_evo,
            mode='lines+markers',
            name=f"{label} (N={n_samples})",
            line=dict(color=color, width=2, dash=dash),
            marker=dict(size=5),
            hovertemplate="a=%{x}<br>β=%{y:.3f}<extra>"+label+"</extra>"
        ))

        # --- Store for programmatic use ---
        beta_curves[(vol_key, approach)] = dict(
            a_values=a_values,
            betas=betas_evo,
            alpha_global=alpha_global,
            n_samples=n_samples,
            n_points=len(points_df)
        )

        # --- Console summary ---
        if len(betas_evo) > 0:
            b1 = betas_evo[0]
            bL = betas_evo[-1]
            print(f"[OK] {vol_cfg['label']} {approach_labels[approach]} "
                  f"| series={n_samples}, points={len(points_df)} "
                  f"| β(a=1)={np.round(b1,3) if np.isfinite(b1) else 'nan'}, "
                  f"β(a=max)={np.round(bL,3) if np.isfinite(bL) else 'nan'}")

# --- Theoretical β=0.5 line ---
if len(fig.data) > 0:
    # Determine x-range union
    max_len = 0
    for tr in fig.data:
        max_len = max(max_len, len(tr.x))
    if max_len > 0:
        x_line = list(range(1, max_len + 1))
        fig.add_trace(go.Scatter(
            x=x_line, y=[0.5]*len(x_line), mode='lines',
            name='Theoretical β = 0.5', line=dict(color='black', dash='dash', width=2)
        ))

# --- Layout ---
fig.update_layout(
    title=dict(text=f"Beta Evolution (buy+sell combined) • Method: {method_label}", font=dict(size=18)),
    xaxis=dict(title='a (insertion threshold)'),
    yaxis=dict(title='β (slope)', range=[0.0, 1.0]),
    width=1100, height=700, template='plotly_white',
    legend=dict(orientation="v", yanchor="bottom", y=0.01, xanchor="right", x=0.99, font=dict(size=12))
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.25)')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.25)')

fig.show()

print("\nPlotted β(a) for all available (volume × approach). Color = volume, line style = approach (solid=Gen, dash=Heur, dot=Hist).")
print("Objects available: beta_curves[(vol_key, approach)] -> {a_values, betas, alpha_global, n_samples, n_points}")

[OK] V=485 Hist | series=224, points=1904 | β(a=1)=0.784, β(a=max)=0.802
[OK] V=485 Heur | series=224, points=1934 | β(a=1)=0.753, β(a=max)=0.753
[OK] V=485 Gen | series=175, points=1495 | β(a=1)=0.697, β(a=max)=0.675
[OK] V=300 Hist | series=224, points=1885 | β(a=1)=0.764, β(a=max)=0.785
[OK] V=300 Heur | series=224, points=1912 | β(a=1)=0.753, β(a=max)=0.756
[OK] V=300 Gen | series=172, points=1443 | β(a=1)=0.686, β(a=max)=0.666
[OK] V=75 Hist | series=223, points=1785 | β(a=1)=0.674, β(a=max)=0.697
[OK] V=75 Heur | series=223, points=1809 | β(a=1)=0.665, β(a=max)=0.69
[OK] V=75 Gen | series=168, points=1316 | β(a=1)=0.636, β(a=max)=0.621



Plotted β(a) for all available (volume × approach). Color = volume, line style = approach (solid=Gen, dash=Heur, dot=Hist).
Objects available: beta_curves[(vol_key, approach)] -> {a_values, betas, alpha_global, n_samples, n_points}


In [13]:
# Create an interactive plotly widget to switch between insertion points with all estimation methods shown simultaneously
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

# First, prepare data for all insertions
# Group by sample_id and create insertion numbers based on order
points_df_with_insertion = points_df.copy()
points_df_with_insertion['insertion_number'] = points_df_with_insertion.groupby('sample_id').cumcount() + 1
max_insertions = points_df_with_insertion['insertion_number'].max()

# Calculate alpha_global
alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

# Available estimation methods
estimation_methods = ['ols', 'huber', 'lad', 'ratio-median', 'ratio-trim', 'deming']
method_colors = {
    'ols': 'red',
    'huber': 'blue', 
    'lad': 'orange',
    'ratio-median': 'purple',
    'ratio-trim': 'brown',
    'deming': 'pink'
}

# Prepare data for each insertion with all methods
insertion_data_dict = {}
for insertion_num in range(1, max_insertions + 1):
    insertion_data = points_df_with_insertion[points_df_with_insertion['insertion_number'] == insertion_num]
    
    if len(insertion_data) > 0:
        X = insertion_data["x"].to_numpy()
        Y = insertion_data["y"].to_numpy()
        sample_ids = insertion_data["sample_id"].to_numpy()
        
        # Filter out invalid points
        valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        x_valid = X[valid_mask]
        y_valid = Y[valid_mask]
        
        # Calculate fitted beta for each method
        beta_fitted_dict = {}
        for method in estimation_methods:
            if len(x_valid) >= 2:
                beta_fitted_dict[method] = beta_fit(x_valid, y_valid, alpha_global, method=method)
            else:
                beta_fitted_dict[method] = np.nan
        
        insertion_data_dict[insertion_num] = {
            'x_all': X,
            'y_all': Y,
            'sample_ids': sample_ids,
            'x_valid': x_valid,
            'y_valid': y_valid,
            'beta_fitted_dict': beta_fitted_dict,
            'n_points': len(X),
            'n_valid': len(x_valid)
        }

# Create FigureWidget for interactive updates
fig = go.FigureWidget()

# Create widgets
insertion_dropdown = widgets.Dropdown(
    options=[(f'Insertion {i}', i) for i in range(1, max_insertions + 1)],
    value=1,
    description='Insertion:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

# Arrow control buttons
prev_button = widgets.Button(
    description='◀ Previous',
    button_style='info',
    layout=widgets.Layout(width='100px')
)

next_button = widgets.Button(
    description='Next ▶',
    button_style='info',
    layout=widgets.Layout(width='100px')
)

# Create output widget for statistics
stats_output = widgets.Output()

def update_plot(insertion_num):
    if insertion_num in insertion_data_dict:
        data = insertion_data_dict[insertion_num]
        
        # Clear existing traces
        with fig.batch_update():
            fig.data = []
            
            # Separate points by buy/sell based on sample_id
            buy_mask = data['sample_ids'] < 1000000
            sell_mask = data['sample_ids'] >= 1000000
            
            # Add scatter plot for buy orders (blue/green)
            if np.any(buy_mask):
                fig.add_scatter(
                    x=data['x_all'][buy_mask],
                    y=data['y_all'][buy_mask],
                    mode='markers',
                    name=f'Buy orders (n={np.sum(buy_mask)})',
                    marker=dict(size=8, opacity=0.7, color='green'),
                    hovertemplate='x=%{x:.6f}<br>y=%{y:.6f}<extra></extra>'
                )
            
            # Add scatter plot for sell orders (red)
            if np.any(sell_mask):
                fig.add_scatter(
                    x=data['x_all'][sell_mask],
                    y=data['y_all'][sell_mask],
                    mode='markers',
                    name=f'Sell orders (n={np.sum(sell_mask)})',
                    marker=dict(size=8, opacity=0.7, color='red'),
                    hovertemplate='x=%{x:.6f}<br>y=%{y:.6f}<extra></extra>'
                )
            
            # Add fitted lines for all methods
            if len(data['x_valid']) >= 2 and len(data['x_all']) > 0:
                x_range = np.linspace(data['x_all'].min(), data['x_all'].max(), 100)
                
                for method in estimation_methods:
                    beta_fitted = data['beta_fitted_dict'][method]
                    if np.isfinite(beta_fitted):
                        y_fitted = alpha_global + beta_fitted * x_range
                        fig.add_scatter(
                            x=x_range,
                            y=y_fitted,
                            mode='lines',
                            name=f'{method.upper()}: β = {beta_fitted:.6f}',
                            line=dict(color=method_colors[method], dash='dash', width=2)
                        )
            
            # Add theoretical line
            beta_theory = 0.5
            if len(data['x_all']) > 0:
                x_range = np.linspace(data['x_all'].min(), data['x_all'].max(), 100)
                y_theory = alpha_global + beta_theory * x_range
                fig.add_scatter(
                    x=x_range,
                    y=y_theory,
                    mode='lines',
                    name=f'Theoretical: β = {beta_theory:.6f}',
                    line=dict(color='black', dash='dot', width=3)
                )
            
            # Update layout
            fig.update_layout(
                title=f'Market Impact: Insertion {insertion_num} (All Estimation Methods)',
                xaxis_title='log(Q / V_exp)',
                yaxis_title='log(Impact)',
                template='plotly_white',
                width=1200,
                height=700,
                legend=dict(
                    font=dict(size=11),
                    orientation="v",
                    yanchor="top",
                    y=0.99,
                    xanchor="left",
                    x=0.01
                ),
                showlegend=True
            )
            
            # Add grid
            fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
            fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
        
        # Update statistics output
        with stats_output:
            stats_output.clear_output()
            print(f"Insertion {insertion_num} Statistics:")
            print(f"Alpha (fixed intercept): {alpha_global:.6f}")
            print(f"Theoretical Beta: {beta_theory:.6f}")
            print(f"Total points: {data['n_points']}")
            print(f"Valid points: {data['n_valid']}")
            print(f"Buy orders: {np.sum(data['sample_ids'] < 1000000)}")
            print(f"Sell orders: {np.sum(data['sample_ids'] >= 1000000)}")
            print()
            print("Fitted Beta values by method:")
            for method in estimation_methods:
                beta_fitted = data['beta_fitted_dict'][method]
                print(f"  {method.upper():12}: β = {beta_fitted:.6f}")

# Connect widgets to update function
def on_dropdown_change(change):
    update_plot(insertion_dropdown.value)

def on_prev_click(b):
    current_val = insertion_dropdown.value
    if current_val > 1:
        insertion_dropdown.value = current_val - 1

def on_next_click(b):
    current_val = insertion_dropdown.value
    if current_val < max_insertions:
        insertion_dropdown.value = current_val + 1

insertion_dropdown.observe(on_dropdown_change, names='value')
prev_button.on_click(on_prev_click)
next_button.on_click(on_next_click)

# Initialize with first insertion
update_plot(1)

# Create control panel layout
controls_row = widgets.HBox([prev_button, insertion_dropdown, next_button])
controls = widgets.VBox([controls_row])

# Display widgets and plot
display(widgets.VBox([controls, stats_output, fig]))


In [14]:
# === One-cell interactive plot: all experiments on one scatter + all methods' fits, by insertion ===
# Assumes available in the notebook:
#   - volume_experiments, approach_labels (as in your snippet)
#   - functions: compute_points_and_coeffs, calculate_impact, calculate_market_volume, beta_fit
#   - CSV: /app/batches_equal_sample_day_map.csv
#
# Conventions in this plot:
#   - Point color = volume (V=485/300/75)
#   - Point marker symbol = approach (Hist/Heur/Gen)
#   - Fitted-line dash = estimation method (OLS/Huber/LAD/RatioMedian/RatioTrim/Deming)
#   - Lines use same color as their volume to keep "1 volume = 1 color"

import os
import pickle
import yaml
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ---------- Styling maps ----------
volume_color = {vk: cfg["color"] for vk, cfg in volume_experiments.items()}
approach_marker = {"hist": "circle", "heur": "square", "gen": "diamond"}
approach_labels = approach_labels if 'approach_labels' in globals() else {"hist": "Hist", "heur": "Heur", "gen": "Gen"}

# Distinct dashes per method (lines)
estimation_methods = ['ols', 'huber', 'lad', 'ratio-median', 'ratio-trim', 'deming']
method_dash = {
    'ols': 'solid',
    'huber': 'dash',
    'lad': 'dot',
    'ratio-median': 'dashdot',
    'ratio-trim': 'longdash',
    'deming': 'longdashdot'
}

# ---------- Load base day map once ----------
BASE_DAY_MAP_PATH = '/app/batches_equal_sample_day_map.csv'
base_day_map = pd.read_csv(BASE_DAY_MAP_PATH)

# ---------- Helper: load & compute points for (volume, approach) ----------
def load_points_for_experiment(vol_key, approach_key):
    cfg = volume_experiments[vol_key]["approaches"][approach_key]
    buy_dir  = f'/app/data_saved/{cfg["buy"]}'
    sell_dir = f'/app/data_saved/{cfg["sell"]}'
    # dict paths
    b_buy_pkl  = os.path.join(buy_dir,  'filtered_b_dict.pkl')
    b_sell_pkl = os.path.join(sell_dir, 'filtered_b_dict.pkl')
    m_buy_pkl  = os.path.join(buy_dir,  'filtered_m_dict.pkl')
    m_sell_pkl = os.path.join(sell_dir, 'filtered_m_dict.pkl')
    # config
    cfg_path   = os.path.join(buy_dir, 'used_config.yaml')

    # load dicts
    with open(b_buy_pkl, 'rb')  as f: b_buy  = pickle.load(f)
    with open(b_sell_pkl, 'rb') as f: b_sell = pickle.load(f)
    with open(m_buy_pkl, 'rb')  as f: m_buy  = pickle.load(f)
    with open(m_sell_pkl, 'rb') as f: m_sell = pickle.load(f)

    # read config
    try:
        with open(cfg_path, 'r') as f:
            c = yaml.safe_load(f)
        num_insertions     = int(c["num_insertions"])
        midprice_step_size = int(c["midprice_step_size"])
        hist_msgs          = int(c["n_messages"])
        n_gen_msgs         = int(c["n_gen_msgs"])
        tick_size          = int(c.get("tick_size", 100))
        hist_steps = hist_msgs // midprice_step_size
        gen_steps  = n_gen_msgs // midprice_step_size
        gen_block  = gen_steps + 1
    except Exception:
        # fallbacks (should rarely happen)
        num_insertions, tick_size = 10, 100
        hist_steps, gen_steps = 500, 50
        gen_block = gen_steps + 1

    # combine buy+sell with offset for sell
    OFFSET = 1_000_000
    m_combined, b_combined = {}, {}
    for k, v in m_buy.items():  m_combined[int(k)] = v
    for k, v in b_buy.items():  b_combined[int(k)] = v
    for k, v in m_sell.items(): m_combined[int(k) + OFFSET] = v
    for k, v in b_sell.items(): b_combined[int(k) + OFFSET] = v

    # build combined day-map (global variable consumed by compute_points_and_coeffs)
    global sample_day_map
    dm_sell = base_day_map.copy()
    dm_sell["sample_id"] = dm_sell["sample_id"].astype(int) + OFFSET
    sample_day_map = pd.concat([base_day_map, dm_sell], ignore_index=True)

    # compute points & per-sample alphas
    points_df, coeffs_df = compute_points_and_coeffs(
        b_seq_inp=b_combined,
        msg_seq_raw=m_combined,
        hist_steps=hist_steps,
        gen_block=gen_block,
        num_insertions=num_insertions,
        tick_size=tick_size
    )
    # alpha_global for this (vol,approach)
    alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

    # tag metadata
    points_df = points_df.copy()
    points_df["vol_key"] = vol_key
    points_df["vol_label"] = volume_experiments[vol_key]["label"]
    points_df["approach"] = approach_key
    points_df["approach_label"] = approach_labels[approach_key]
    return points_df, alpha_global, num_insertions

# ---------- Aggregate all experiments ----------
results = {}   # (vol_key, approach) -> dict(points_df, alpha, max_ins)
max_insertions_global = 1
total_groups = 0

for vol_key in volume_experiments.keys():
    for approach_key in ["hist", "heur", "gen"]:
        try:
            pts, a_glob, n_ins = load_points_for_experiment(vol_key, approach_key)
            if pts.empty:
                print(f"[SKIP] {volume_experiments[vol_key]['label']} {approach_labels[approach_key]}: no points")
                continue
            results[(vol_key, approach_key)] = dict(points=pts, alpha=a_glob, max_ins=n_ins)
            max_insertions_global = max(max_insertions_global, int(pts["insertion"].max()))
            total_groups += 1
            print(f"[OK] {volume_experiments[vol_key]['label']} {approach_labels[approach_key]}: "
                  f"samples={pts['sample_id'].nunique()}, points={len(pts)}")
        except Exception as e:
            print(f"[SKIP] {vol_key}-{approach_key}: {e}")

if total_groups == 0:
    raise RuntimeError("No experiment data loaded; check paths and files.")

# ---------- Precompute global x-range per insertion for the theory line ----------
global_x_range_per_ins = {}
for ins in range(1, max_insertions_global + 1):
    xs = []
    for d in results.values():
        sub = d["points"][d["points"]["insertion"] == ins]
        if not sub.empty:
            xv = sub["x"].to_numpy()
            xv = xv[np.isfinite(xv)]
            if xv.size > 0:
                xs.append([np.min(xv), np.max(xv)])
    if xs:
        xmin = min(v[0] for v in xs)
        xmax = max(v[1] for v in xs)
        if np.isfinite(xmin) and np.isfinite(xmax) and xmax > xmin:
            global_x_range_per_ins[ins] = (xmin, xmax)

# ---------- Interactive FigureWidget ----------
fig = go.FigureWidget()

insertion_dropdown = widgets.Dropdown(
    options=[(f'Insertion {i}', i) for i in range(1, max_insertions_global + 1)],
    value=1, description='Insertion:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)
prev_button = widgets.Button(description='◀ Previous', button_style='info', layout=widgets.Layout(width='110px'))
next_button = widgets.Button(description='Next ▶',     button_style='info', layout=widgets.Layout(width='110px'))

stats_output = widgets.Output()

def fit_beta(x_valid, y_valid, alpha, method):
    # Safe wrapper around beta_fit
    try:
        if x_valid.size >= 2:
            return float(beta_fit(x_valid, y_valid, alpha, method=method))
    except Exception:
        return np.nan
    return np.nan

def update_plot(insertion_num: int):
    with fig.batch_update():
        fig.data = []

        # Draw points & fitted lines for each (volume, approach)
        for (vol_key, approach_key), dat in results.items():
            pts = dat["points"]
            alpha = dat["alpha"]
            sub = pts[pts["insertion"] == insertion_num]
            if sub.empty:
                continue

            # Prepare point aesthetics
            color = volume_color.get(vol_key, 'gray')
            symbol = approach_marker.get(approach_key, 'circle')
            label_points = f"{volume_experiments[vol_key]['label']} {approach_labels[approach_key]}"

            # Add scatter points (buy+sell already combined)
            X = sub["x"].to_numpy()
            Y = sub["y"].to_numpy()
            valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
            x_valid = X[valid_mask]
            y_valid = Y[valid_mask]

            fig.add_scatter(
                x=X, y=Y, mode='markers',
                name=f"{label_points} • points (n={len(X)})",
                marker=dict(size=8, opacity=0.7, color=color, symbol=symbol, line=dict(width=0.5, color='rgba(0,0,0,0.35)')),
                hovertemplate='x=%{x:.6f}<br>y=%{y:.6f}<extra>'+label_points+'</extra>'
            )

            # Fitted lines for all methods (same volume color, method = dash)
            if x_valid.size >= 2:
                if insertion_num in global_x_range_per_ins:
                    x0, x1 = global_x_range_per_ins[insertion_num]
                else:
                    x0, x1 = np.min(x_valid), np.max(x_valid)
                if np.isfinite(x0) and np.isfinite(x1) and x1 > x0:
                    x_line = np.linspace(x0, x1, 120)
                    for method in estimation_methods:
                        beta = fit_beta(x_valid, y_valid, alpha, method)
                        if np.isfinite(beta):
                            y_line = alpha + beta * x_line
                            fig.add_scatter(
                                x=x_line, y=y_line, mode='lines',
                                name=f"{label_points} • {method.upper()} (β={beta:.3f})",
                                line=dict(color=color, dash=method_dash[method], width=2),
                                hovertemplate='β='+f'{beta:.6f}'+'<extra>'+label_points+' • '+method.upper()+'</extra>'
                            )

        # Theoretical β=0.5 line over global x-range for this insertion
        if insertion_num in global_x_range_per_ins:
            x0, x1 = global_x_range_per_ins[insertion_num]
            x_theory = np.linspace(x0, x1, 120)
            # Alpha for theory: we show line families; choose median alpha across groups for this insertion for reference
            alphas = []
            for dat in results.values():
                alphas.append(dat["alpha"])
            alpha_ref = float(np.nanmedian(alphas)) if len(alphas) > 0 else 0.0
            y_theory = alpha_ref + 0.5 * x_theory
            fig.add_scatter(
                x=x_theory, y=y_theory, mode='lines',
                name='Theoretical β = 0.5', line=dict(color='black', dash='dot', width=3)
            )

        # Layout
        fig.update_layout(
            title=f"Market Impact • Insertion {insertion_num} • All volumes & approaches (points) + all methods (fits)",
            xaxis_title='log(Q / V_exp)',
            yaxis_title='log(Impact)',
            template='plotly_white',
            width=1200, height=750,
            legend=dict(font=dict(size=11), orientation="v", yanchor="top", y=0.99, xanchor="left", x=0.01),
            showlegend=True
        )
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.28)')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.28)')

    # Stats panel
    with stats_output:
        stats_output.clear_output()
        print(f"Insertion {insertion_num} — summary per group:")
        for (vol_key, approach_key), dat in results.items():
            pts = dat["points"]
            alpha = dat["alpha"]
            sub = pts[pts["insertion"] == insertion_num]
            if sub.empty: 
                continue
            X = sub["x"].to_numpy(); Y = sub["y"].to_numpy()
            valid = np.isfinite(X) & np.isfinite(Y) & (X != 0)
            x_valid, y_valid = X[valid], Y[valid]
            print(f"  {volume_experiments[vol_key]['label']} {approach_labels[approach_key]}: "
                  f"points={len(X)}, valid={len(x_valid)}, alpha={alpha:.4f}")
            # list betas
            for m in estimation_methods:
                b = fit_beta(x_valid, y_valid, alpha, m)
                if np.isfinite(b):
                    print(f"    {m.upper():12} β={b:.4f}")
        print("Legend: point color = volume; point symbol = approach; line dash = method; line color = volume.")

# ---------- Wire controls ----------
def on_dropdown_change(change):
    if change['name'] == 'value':
        update_plot(change['new'])

def on_prev_click(_):
    v = insertion_dropdown.value
    if v > 1:
        insertion_dropdown.value = v - 1

def on_next_click(_):
    v = insertion_dropdown.value
    if v < max_insertions_global:
        insertion_dropdown.value = v + 1

insertion_dropdown.observe(on_dropdown_change, names='value')
prev_button.on_click(on_prev_click)
next_button.on_click(on_next_click)

# ---------- Initial render ----------
update_plot(insertion_dropdown.value)

controls = widgets.HBox([prev_button, insertion_dropdown, next_button])
display(widgets.VBox([controls, stats_output, fig]))

[OK] V=485 Hist: samples=224, points=1904
[OK] V=485 Heur: samples=224, points=1934
[OK] V=485 Gen: samples=175, points=1495
[OK] V=300 Hist: samples=224, points=1885
[OK] V=300 Heur: samples=224, points=1912
[OK] V=300 Gen: samples=172, points=1443
[OK] V=75 Hist: samples=223, points=1785
[OK] V=75 Heur: samples=223, points=1809
[OK] V=75 Gen: samples=168, points=1316


In [15]:
# === Interactive: volume dropdown + insertion navigation; methods as colors ===
# Assumes present:
#   - volume_experiments, approach_labels (from your snippet)
#   - functions: compute_points_and_coeffs, calculate_impact, calculate_market_volume, beta_fit
#   - CSV: /app/batches_equal_sample_day_map.csv
#
# Behavior:
#   - Dropdown selects volume (V=485/300/75)
#   - Points: all approaches (Hist/Heur/Gen) together, marker symbol encodes approach
#   - Fits: one line per estimation method (color = method), fitted on ALL approaches' points for the selected volume
#   - Buy+Sell merged BEFORE regression (sell sample_id offset +1_000_000)

import os
import pickle
import yaml
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ---------- Palettes & mappings ----------
# Marker symbol per approach
approach_marker = {"hist": "circle", "heur": "square", "gen": "diamond"}
approach_labels = approach_labels if 'approach_labels' in globals() else {"hist": "Hist", "heur": "Heur", "gen": "Gen"}

# Estimation methods and their COLORS (lines)
estimation_methods = ['ols', 'huber', 'lad', 'ratio-median', 'ratio-trim', 'deming']
method_colors = {
    'ols': 'red',
    'huber': 'blue',
    'lad': 'orange',
    'ratio-median': 'purple',
    'ratio-trim': 'brown',
    'deming': 'pink'
}

# Neutral color for points (volume is fixed here; methods use colors on lines)
POINT_COLOR = 'rgba(60,60,60,0.85)'

# ---------- Load base day-map once ----------
BASE_DAY_MAP_PATH = '/app/batches_equal_sample_day_map.csv'
base_day_map = pd.read_csv(BASE_DAY_MAP_PATH)

# ---------- Helper: load & compute points for (volume, approach) ----------
def load_points_for_experiment(vol_key, approach_key):
    cfg = volume_experiments[vol_key]["approaches"][approach_key]
    buy_dir  = f'/app/data_saved/{cfg["buy"]}'
    sell_dir = f'/app/data_saved/{cfg["sell"]}'
    # dict paths
    b_buy_pkl  = os.path.join(buy_dir,  'filtered_b_dict.pkl')
    b_sell_pkl = os.path.join(sell_dir, 'filtered_b_dict.pkl')
    m_buy_pkl  = os.path.join(buy_dir,  'filtered_m_dict.pkl')
    m_sell_pkl = os.path.join(sell_dir, 'filtered_m_dict.pkl')
    # config (read from buy side)
    cfg_path   = os.path.join(buy_dir, 'used_config.yaml')

    # load dicts
    with open(b_buy_pkl, 'rb')  as f: b_buy  = pickle.load(f)
    with open(b_sell_pkl, 'rb') as f: b_sell = pickle.load(f)
    with open(m_buy_pkl, 'rb')  as f: m_buy  = pickle.load(f)
    with open(m_sell_pkl, 'rb') as f: m_sell = pickle.load(f)

    # read config
    try:
        with open(cfg_path, 'r') as f:
            c = yaml.safe_load(f)
        num_insertions     = int(c["num_insertions"])
        midprice_step_size = int(c["midprice_step_size"])
        hist_msgs          = int(c["n_messages"])
        n_gen_msgs         = int(c["n_gen_msgs"])
        tick_size          = int(c.get("tick_size", 100))
        hist_steps = hist_msgs // midprice_step_size
        gen_steps  = n_gen_msgs // midprice_step_size
        gen_block  = gen_steps + 1
    except Exception:
        # conservative fallbacks
        num_insertions, tick_size = 10, 100
        hist_steps, gen_steps = 500, 50
        gen_block = gen_steps + 1

    # combine buy+sell with offset for sell
    OFFSET = 1_000_000
    m_combined, b_combined = {}, {}
    for k, v in m_buy.items():  m_combined[int(k)] = v
    for k, v in b_buy.items():  b_combined[int(k)] = v
    for k, v in m_sell.items(): m_combined[int(k) + OFFSET] = v
    for k, v in b_sell.items(): b_combined[int(k) + OFFSET] = v

    # build combined day-map (global for compute_points_and_coeffs)
    global sample_day_map
    dm_sell = base_day_map.copy()
    dm_sell["sample_id"] = dm_sell["sample_id"].astype(int) + OFFSET
    sample_day_map = pd.concat([base_day_map, dm_sell], ignore_index=True)

    # compute points & per-sample alpha
    points_df, coeffs_df = compute_points_and_coeffs(
        b_seq_inp=b_combined,
        msg_seq_raw=m_combined,
        hist_steps=hist_steps,
        gen_block=gen_block,
        num_insertions=num_insertions,
        tick_size=tick_size
    )
    alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

    # tag metadata
    points_df = points_df.copy()
    points_df["vol_key"] = vol_key
    points_df["vol_label"] = volume_experiments[vol_key]["label"]
    points_df["approach"] = approach_key
    points_df["approach_label"] = approach_labels[approach_key]
    return points_df, alpha_global, num_insertions

# ---------- Load ALL (volume, approach) once; cache results ----------
results = {}   # (vol_key, approach) -> dict(points, alpha, max_ins)
max_insertions_global_per_vol = {}  # vol_key -> max insertions observed
for vol_key in volume_experiments.keys():
    max_ins_v = 1
    for approach_key in ["hist", "heur", "gen"]:
        try:
            pts, a_glob, n_ins = load_points_for_experiment(vol_key, approach_key)
            if pts.empty:
                print(f"[SKIP] {volume_experiments[vol_key]['label']} {approach_labels[approach_key]}: no points")
                continue
            results[(vol_key, approach_key)] = dict(points=pts, alpha=a_glob, max_ins=n_ins)
            max_ins_v = max(max_ins_v, int(pts["insertion"].max()))
            print(f"[OK] {volume_experiments[vol_key]['label']} {approach_labels[approach_key]}: "
                  f"samples={pts['sample_id'].nunique()}, points={len(pts)}")
        except Exception as e:
            print(f"[SKIP] {vol_key}-{approach_key}: {e}")
    max_insertions_global_per_vol[vol_key] = max_ins_v

if len(results) == 0:
    raise RuntimeError("No experiment data loaded; check paths/files.")

# ---------- Build volume dropdown options (labels) ----------
vol_options = []
for vk, cfg in volume_experiments.items():
    vol_options.append((cfg["label"], vk))
# default first volume
default_vol_key = vol_options[0][1]

# ---------- Precompute global x-range per (volume, insertion) ----------
vol_ins_xrange = {}  # (vol_key, insertion) -> (xmin, xmax)
for vk in volume_experiments.keys():
    max_ins = max_insertions_global_per_vol[vk]
    for ins in range(1, max_ins + 1):
        xs = []
        for (vol_key, approach_key), dat in results.items():
            if vol_key != vk:
                continue
            sub = dat["points"][dat["points"]["insertion"] == ins]
            if not sub.empty:
                xv = sub["x"].to_numpy()
                xv = xv[np.isfinite(xv)]
                if xv.size > 0:
                    xs.append([np.min(xv), np.max(xv)])
        if xs:
            xmin = min(v[0] for v in xs); xmax = max(v[1] for v in xs)
            if np.isfinite(xmin) and np.isfinite(xmax) and xmax > xmin:
                vol_ins_xrange[(vk, ins)] = (xmin, xmax)

# ---------- Figure + controls ----------
fig = go.FigureWidget()

volume_dropdown = widgets.Dropdown(
    options=vol_options, value=default_vol_key,
    description='Volume:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)

insertion_dropdown = widgets.Dropdown(
    options=[(f'Insertion {i}', i) for i in range(1, max_insertions_global_per_vol[default_vol_key] + 1)],
    value=1, description='Insertion:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)

prev_button = widgets.Button(description='◀ Prev ins', button_style='info', layout=widgets.Layout(width='110px'))
next_button = widgets.Button(description='Next ins ▶', button_style='info', layout=widgets.Layout(width='110px'))

stats_output = widgets.Output()

def fit_beta(x_valid, y_valid, alpha, method):
    try:
        if x_valid.size >= 2:
            return float(beta_fit(x_valid, y_valid, alpha, method=method))
    except Exception:
        return np.nan
    return np.nan

def update_plot(vol_key: str, insertion_num: int):
    with fig.batch_update():
        fig.data = []

        # Gather all approaches for this volume
        groups = [(k[1], v) for k, v in results.items() if k[0] == vol_key]
        if len(groups) == 0:
            fig.update_layout(title="No data for selected volume")
            return

        # Collect ALL points across approaches for the selected insertion
        X_all, Y_all, A_all = [], [], []  # x, y, per-approach alpha
        for approach_key, dat in groups:
            pts = dat["points"]
            alpha = dat["alpha"]
            sub = pts[pts["insertion"] == insertion_num]
            if sub.empty:
                continue

            # Points per approach (markers)
            X = sub["x"].to_numpy()
            Y = sub["y"].to_numpy()
            fig.add_scatter(
                x=X, y=Y, mode='markers',
                name=f"{approach_labels[approach_key]} • points (n={len(X)})",
                marker=dict(
                    size=8, opacity=0.75, color=POINT_COLOR,
                    symbol=approach_marker.get(approach_key, 'circle'),
                    line=dict(width=0.6, color='rgba(0,0,0,0.35)')
                ),
                hovertemplate='x=%{x:.6f}<br>y=%{y:.6f}<extra>'+approach_labels[approach_key]+'</extra>'
            )

            # collect for combined fitting
            X_all.append(X); Y_all.append(Y); A_all.append(alpha)

        if len(X_all) == 0:
            fig.update_layout(title=f"{volume_experiments[vol_key]['label']} • Insertion {insertion_num}: no points")
            return

        X_all = np.concatenate(X_all)
        Y_all = np.concatenate(Y_all)
        # robust reference alpha: median across approaches for this volume
        alpha_ref = float(np.nanmedian(np.array(A_all))) if len(A_all) > 0 else 0.0

        # Valid mask
        valid = np.isfinite(X_all) & np.isfinite(Y_all) & (X_all != 0)
        x_valid = X_all[valid]; y_valid = Y_all[valid]

        # X-range for this (volume, insertion)
        if (vol_key, insertion_num) in vol_ins_xrange:
            x0, x1 = vol_ins_xrange[(vol_key, insertion_num)]
        else:
            x0, x1 = (np.min(x_valid), np.max(x_valid)) if x_valid.size > 0 else (0.0, 1.0)
        x_line = np.linspace(x0, x1, 160) if np.isfinite(x0) and np.isfinite(x1) and x1 > x0 else None

        # Fitted lines: one per METHOD (color = method)
        if x_valid.size >= 2 and x_line is not None:
            for method in estimation_methods:
                beta = fit_beta(x_valid, y_valid, alpha_ref, method)
                if np.isfinite(beta):
                    y_line = alpha_ref + beta * x_line
                    fig.add_scatter(
                        x=x_line, y=y_line, mode='lines',
                        name=f"{method.upper()} (β={beta:.3f})",
                        line=dict(color=method_colors[method], width=3),
                        hovertemplate='β='+f'{beta:.6f}'+'<extra>'+method.upper()+'</extra>'
                    )

        # Theoretical β=0.5 line
        if x_line is not None:
            y_theory = alpha_ref + 0.5 * x_line
            fig.add_scatter(
                x=x_line, y=y_theory, mode='lines',
                name='Theoretical β = 0.5', line=dict(color='black', dash='dot', width=2.5)
            )

        # Layout
        fig.update_layout(
            title=f"{volume_experiments[vol_key]['label']} • Insertion {insertion_num} • Points: approaches | Lines: methods (colors)",
            xaxis_title='log(Q / V_exp)',
            yaxis_title='log(Impact)',
            template='plotly_white',
            width=1200, height=750,
            legend=dict(font=dict(size=11), orientation="v", yanchor="top", y=0.99, xanchor="left", x=0.01),
            showlegend=True
        )
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.28)')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.28)')

    # Stats panel
    with stats_output:
        stats_output.clear_output()
        print(f"{volume_experiments[vol_key]['label']} — Insertion {insertion_num}")
        print(f"Alpha (reference, median across approaches): {alpha_ref:.6f}")
        print("Approach point counts:")
        for approach_key, dat in groups:
            sub = dat["points"][dat["points"]["insertion"] == insertion_num]
            if sub.empty: 
                continue
            print(f"  {approach_labels[approach_key]:>4}: n={len(sub)}")
        if x_valid.size >= 2:
            print("Fitted β by method:")
            for m in estimation_methods:
                b = fit_beta(x_valid, y_valid, alpha_ref, m)
                if np.isfinite(b):
                    print(f"  {m.upper():12} β={b:.4f}")
        else:
            print("Not enough valid points to fit β.")

# ---------- Wiring controls ----------
def on_volume_change(change):
    if change['name'] == 'value':
        vk = change['new']
        # reset insertion options for this volume
        insertion_dropdown.options = [(f'Insertion {i}', i) for i in range(1, max_insertions_global_per_vol[vk] + 1)]
        insertion_dropdown.value = 1  # triggers update via its observer

def on_insertion_change(change):
    if change['name'] == 'value':
        update_plot(volume_dropdown.value, change['new'])

def on_prev_click(_):
    v = insertion_dropdown.value
    if v > 1:
        insertion_dropdown.value = v - 1

def on_next_click(_):
    v = insertion_dropdown.value
    vmax = max_insertions_global_per_vol[volume_dropdown.value]
    if v < vmax:
        insertion_dropdown.value = v + 1

volume_dropdown.observe(on_volume_change, names='value')
insertion_dropdown.observe(on_insertion_change, names='value')
prev_button.on_click(on_prev_click)
next_button.on_click(on_next_click)

# ---------- Initial render ----------
update_plot(volume_dropdown.value, insertion_dropdown.value)
controls = widgets.HBox([volume_dropdown, prev_button, insertion_dropdown, next_button])
display(widgets.VBox([controls, stats_output, fig]))

[OK] V=485 Hist: samples=224, points=1904
[OK] V=485 Heur: samples=224, points=1934
[OK] V=485 Gen: samples=175, points=1495
[OK] V=300 Hist: samples=224, points=1885
[OK] V=300 Heur: samples=224, points=1912
[OK] V=300 Gen: samples=172, points=1443
[OK] V=75 Hist: samples=223, points=1785
[OK] V=75 Heur: samples=223, points=1809
[OK] V=75 Gen: samples=168, points=1316


In [16]:
# Create an interactive plotly widget to show buy and sell samples together on one plot
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

# First, prepare data for all samples
# Get unique sample IDs and separate buy/sell
unique_samples = sorted(points_df['sample_id'].unique())
buy_samples = [s for s in unique_samples if s < 1000000]
sell_samples = [s for s in unique_samples if s >= 1000000]

# Calculate alpha_global
alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

# Prepare data for each sample with detailed message info
sample_data_dict = {}
for sample_id in unique_samples:
    sample_data = points_df[points_df['sample_id'] == sample_id].copy()
    
    if len(sample_data) > 0:
        # Sort by insertion order (assuming data is already ordered)
        sample_data = sample_data.sort_index()
        
        X = sample_data["x"].to_numpy()
        Y = sample_data["y"].to_numpy()
        
        # Get detailed message info for hover
        hover_info = []
        volumes = []
        message_ids = []
        insertion_numbers = []
        
        for idx, (_, row) in enumerate(sample_data.iterrows()):
            insertion_num = idx + 1
            insertion_numbers.append(insertion_num)
            
            if sample_id in m_dict_combined:
                messages = m_dict_combined[sample_id]
                # Find insertion points (messages with id 77777777)
                insertion_points = messages[messages[:, 0] == 77777777]
                
                # Get the volume for this insertion (idx because 0-indexed)
                if idx < len(insertion_points):
                    volume = insertion_points[idx, 5]  # 5th index is volume
                    msg_id = insertion_points[idx, 0]  # 0th index is message id
                    volumes.append(volume)
                    message_ids.append(msg_id)
                    
                    # Create detailed hover text
                    hover_text = (f"Sample ID: {sample_id}<br>"
                                f"Insertion: {insertion_num}<br>"
                                f"Message ID: {msg_id}<br>"
                                f"Volume: {volume}<br>"
                                f"x (log(Q/V_exp)): {X[idx]:.6f}<br>"
                                f"y (log(Impact)): {Y[idx]:.6f}")
                    hover_info.append(hover_text)
                else:
                    volumes.append(np.nan)
                    message_ids.append(np.nan)
                    hover_info.append(f"Sample ID: {sample_id}<br>Insertion: {insertion_num}<br>No insertion point data")
            else:
                volumes.append(np.nan)
                message_ids.append(np.nan)
                hover_info.append(f"Sample ID: {sample_id}<br>Insertion: {insertion_num}<br>No message data")
        
        # Filter out invalid points for fitting
        valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        x_valid = X[valid_mask]
        y_valid = Y[valid_mask]
        
        # Calculate fitted beta for OLS only
        beta_fitted_ols = np.nan
        if len(x_valid) >= 2:
            beta_fitted_ols = beta_fit(x_valid, y_valid, alpha_global, method='ols')
        
        # Determine if this is a buy or sell order
        order_type = "Buy" if sample_id < 1000000 else "Sell"
        
        sample_data_dict[sample_id] = {
            'x_all': X,
            'y_all': Y,
            'volumes': np.array(volumes),
            'message_ids': np.array(message_ids),
            'insertion_numbers': np.array(insertion_numbers),
            'hover_info': hover_info,
            'x_valid': x_valid,
            'y_valid': y_valid,
            'beta_fitted_ols': beta_fitted_ols,
            'n_points': len(X),
            'n_valid': len(x_valid),
            'order_type': order_type
        }

# Create FigureWidget for interactive updates
fig = go.FigureWidget()

# Create widgets - dropdown for buy samples only
buy_dropdown = widgets.Dropdown(
    options=[(f'Buy Sample {sample_id}', sample_id) for sample_id in buy_samples],
    value=buy_samples[0] if buy_samples else None,
    description='Buy Sample:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

sell_dropdown = widgets.Dropdown(
    options=[(f'Sell Sample {sample_id}', sample_id) for sample_id in sell_samples],
    value=sell_samples[0] if sell_samples else None,
    description='Sell Sample:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

# Arrow control buttons for buy samples
prev_buy_button = widgets.Button(
    description='◀ Prev Buy',
    button_style='success',
    layout=widgets.Layout(width='100px')
)

next_buy_button = widgets.Button(
    description='Next Buy ▶',
    button_style='success',
    layout=widgets.Layout(width='100px')
)

# Arrow control buttons for sell samples
prev_sell_button = widgets.Button(
    description='◀ Prev Sell',
    button_style='danger',
    layout=widgets.Layout(width='100px')
)

next_sell_button = widgets.Button(
    description='Next Sell ▶',
    button_style='danger',
    layout=widgets.Layout(width='100px')
)

# Create output widget for statistics
stats_output = widgets.Output()

def update_plot(buy_sample_id, sell_sample_id):
    # Clear existing traces
    with fig.batch_update():
        fig.data = []
        
        # Collect all valid points from both active samples for combined fitting
        combined_x_valid = []
        combined_y_valid = []
        
        # Add buy sample data
        if buy_sample_id and buy_sample_id in sample_data_dict:
            buy_data = sample_data_dict[buy_sample_id]
            combined_x_valid.extend(buy_data['x_valid'])
            combined_y_valid.extend(buy_data['y_valid'])
            
            # Add scatter plot for buy sample
            fig.add_scatter(
                x=buy_data['x_all'],
                y=buy_data['y_all'],
                mode='markers+lines',
                name=f'Buy {buy_sample_id} (n={buy_data["n_points"]})',
                marker=dict(size=10, opacity=0.8, color='green'),
                line=dict(color='green', width=2, dash='dot'),
                hovertemplate='%{text}<extra></extra>',
                text=buy_data['hover_info']
            )
            
            # Add fitted line for buy sample OLS
            if len(buy_data['x_valid']) >= 2 and len(buy_data['x_all']) > 0:
                x_range = np.linspace(buy_data['x_all'].min(), buy_data['x_all'].max(), 100)
                beta_fitted = buy_data['beta_fitted_ols']
                if np.isfinite(beta_fitted):
                    y_fitted = alpha_global + beta_fitted * x_range
                    fig.add_scatter(
                        x=x_range,
                        y=y_fitted,
                        mode='lines',
                        name=f'Buy OLS: β = {beta_fitted:.6f}',
                        line=dict(color='darkgreen', dash='dash', width=2)
                    )
        
        # Add sell sample data
        if sell_sample_id and sell_sample_id in sample_data_dict:
            sell_data = sample_data_dict[sell_sample_id]
            combined_x_valid.extend(sell_data['x_valid'])
            combined_y_valid.extend(sell_data['y_valid'])
            
            # Add scatter plot for sell sample
            fig.add_scatter(
                x=sell_data['x_all'],
                y=sell_data['y_all'],
                mode='markers+lines',
                name=f'Sell {sell_sample_id} (n={sell_data["n_points"]})',
                marker=dict(size=10, opacity=0.8, color='red'),
                line=dict(color='red', width=2, dash='dot'),
                hovertemplate='%{text}<extra></extra>',
                text=sell_data['hover_info']
            )
            
            # Add fitted line for sell sample OLS
            if len(sell_data['x_valid']) >= 2 and len(sell_data['x_all']) > 0:
                x_range = np.linspace(sell_data['x_all'].min(), sell_data['x_all'].max(), 100)
                beta_fitted = sell_data['beta_fitted_ols']
                if np.isfinite(beta_fitted):
                    y_fitted = alpha_global + beta_fitted * x_range
                    fig.add_scatter(
                        x=x_range,
                        y=y_fitted,
                        mode='lines',
                        name=f'Sell OLS: β = {beta_fitted:.6f}',
                        line=dict(color='darkred', dash='dash', width=2)
                    )
        
        # Add combined fitted line for both active samples
        if len(combined_x_valid) >= 2:
            combined_x_valid = np.array(combined_x_valid)
            combined_y_valid = np.array(combined_y_valid)
            beta_combined = beta_fit(combined_x_valid, combined_y_valid, alpha_global, method='ols')
            
            if np.isfinite(beta_combined):
                # Get combined x range from both samples
                all_x = []
                if buy_sample_id and buy_sample_id in sample_data_dict:
                    all_x.extend(sample_data_dict[buy_sample_id]['x_all'])
                if sell_sample_id and sell_sample_id in sample_data_dict:
                    all_x.extend(sample_data_dict[sell_sample_id]['x_all'])
                
                if len(all_x) > 0:
                    x_range = np.linspace(min(all_x), max(all_x), 100)
                    y_combined = alpha_global + beta_combined * x_range
                    fig.add_scatter(
                        x=x_range,
                        y=y_combined,
                        mode='lines',
                        name=f'Combined OLS: β = {beta_combined:.6f}',
                        line=dict(color='purple', dash='solid', width=3)
                    )
        
        # Add theoretical line
        beta_theory = 0.5
        # Get combined x range from both samples
        all_x = []
        if buy_sample_id and buy_sample_id in sample_data_dict:
            all_x.extend(sample_data_dict[buy_sample_id]['x_all'])
        if sell_sample_id and sell_sample_id in sample_data_dict:
            all_x.extend(sample_data_dict[sell_sample_id]['x_all'])
        
        if len(all_x) > 0:
            x_range = np.linspace(min(all_x), max(all_x), 100)
            y_theory = alpha_global + beta_theory * x_range
            fig.add_scatter(
                x=x_range,
                y=y_theory,
                mode='lines',
                name=f'Theoretical: β = {beta_theory:.6f}',
                line=dict(color='black', dash='dot', width=3)
            )
        
        # Update layout
        fig.update_layout(
            title=f'Market Impact: Buy Sample {buy_sample_id} vs Sell Sample {sell_sample_id}',
            xaxis_title='log(Q / V_exp)',
            yaxis_title='log(Impact)',
            template='plotly_white',
            width=1200,
            height=700,
            legend=dict(
                font=dict(size=11),
                orientation="v",
                yanchor="top",
                y=0.99,
                xanchor="left",
                x=0.01
            ),
            showlegend=True
        )
        
        # Add grid
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
    
    # Update statistics output
    with stats_output:
        stats_output.clear_output()
        
        # Buy sample statistics
        if buy_sample_id and buy_sample_id in sample_data_dict:
            buy_data = sample_data_dict[buy_sample_id]
            print(f"Buy Sample {buy_sample_id} Statistics:")
            print(f"  OLS Beta: {buy_data['beta_fitted_ols']:.6f}")
            print(f"  Total insertion points: {buy_data['n_points']}")
            print(f"  Valid points for fitting: {buy_data['n_valid']}")
            
            valid_volumes = buy_data['volumes'][np.isfinite(buy_data['volumes'])]
            if len(valid_volumes) > 0:
                print(f"  Volume range: {np.min(valid_volumes):.0f} - {np.max(valid_volumes):.0f}")
                print(f"  Mean volume: {np.mean(valid_volumes):.2f}")
        
        print()
        
        # Sell sample statistics
        if sell_sample_id and sell_sample_id in sample_data_dict:
            sell_data = sample_data_dict[sell_sample_id]
            print(f"Sell Sample {sell_sample_id} Statistics:")
            print(f"  OLS Beta: {sell_data['beta_fitted_ols']:.6f}")
            print(f"  Total insertion points: {sell_data['n_points']}")
            print(f"  Valid points for fitting: {sell_data['n_valid']}")
            
            valid_volumes = sell_data['volumes'][np.isfinite(sell_data['volumes'])]
            if len(valid_volumes) > 0:
                print(f"  Volume range: {np.min(valid_volumes):.0f} - {np.max(valid_volumes):.0f}")
                print(f"  Mean volume: {np.mean(valid_volumes):.2f}")
        
        print()
        
        # Combined statistics
        combined_x_valid = []
        combined_y_valid = []
        if buy_sample_id and buy_sample_id in sample_data_dict:
            combined_x_valid.extend(sample_data_dict[buy_sample_id]['x_valid'])
            combined_y_valid.extend(sample_data_dict[buy_sample_id]['y_valid'])
        if sell_sample_id and sell_sample_id in sample_data_dict:
            combined_x_valid.extend(sample_data_dict[sell_sample_id]['x_valid'])
            combined_y_valid.extend(sample_data_dict[sell_sample_id]['y_valid'])
        
        if len(combined_x_valid) >= 2:
            combined_x_valid = np.array(combined_x_valid)
            combined_y_valid = np.array(combined_y_valid)
            beta_combined = beta_fit(combined_x_valid, combined_y_valid, alpha_global, method='ols')
            print(f"Combined Active Samples Statistics:")
            print(f"  Combined OLS Beta: {beta_combined:.6f}")
            print(f"  Total combined valid points: {len(combined_x_valid)}")
            print()
        
        print(f"Global Statistics:")
        print(f"  Alpha (fixed intercept): {alpha_global:.6f}")
        print(f"  Theoretical Beta: {beta_theory:.6f}")

# Connect widgets to update function
def on_dropdown_change(change):
    update_plot(buy_dropdown.value, sell_dropdown.value)

def on_prev_buy_click(b):
    current_val = buy_dropdown.value
    current_idx = buy_samples.index(current_val)
    if current_idx > 0:
        buy_dropdown.value = buy_samples[current_idx - 1]

def on_next_buy_click(b):
    current_val = buy_dropdown.value
    current_idx = buy_samples.index(current_val)
    if current_idx < len(buy_samples) - 1:
        buy_dropdown.value = buy_samples[current_idx + 1]

def on_prev_sell_click(b):
    current_val = sell_dropdown.value
    current_idx = sell_samples.index(current_val)
    if current_idx > 0:
        sell_dropdown.value = sell_samples[current_idx - 1]

def on_next_sell_click(b):
    current_val = sell_dropdown.value
    current_idx = sell_samples.index(current_val)
    if current_idx < len(sell_samples) - 1:
        sell_dropdown.value = sell_samples[current_idx + 1]

buy_dropdown.observe(on_dropdown_change, names='value')
sell_dropdown.observe(on_dropdown_change, names='value')
prev_buy_button.on_click(on_prev_buy_click)
next_buy_button.on_click(on_next_buy_click)
prev_sell_button.on_click(on_prev_sell_click)
next_sell_button.on_click(on_next_sell_click)

# Initialize with first samples
if buy_samples and sell_samples:
    update_plot(buy_samples[0], sell_samples[0])

# Create control panel layout
buy_controls_row = widgets.HBox([prev_buy_button, buy_dropdown, next_buy_button])
sell_controls_row = widgets.HBox([prev_sell_button, sell_dropdown, next_sell_button])
controls = widgets.VBox([buy_controls_row, sell_controls_row])

# Display widgets and plot
display(widgets.VBox([controls, stats_output, fig]))


# aggregated analysis

In [17]:
# Calculate mean and std for each insertion across all samples
insertion_stats = []

# First, let's check what columns are available in points_df
print("Available columns in points_df:", points_df.columns.tolist())

# Check if we have insertion_number column, if not, we'll need to create it or use a different approach
if 'insertion_number' not in points_df.columns:
    print("Warning: 'insertion_number' column not found in points_df")
    print("Using sample-based analysis instead...")
    
    # Group by sample_id and create insertion numbers based on order
    points_df_with_insertion = points_df.copy()
    points_df_with_insertion['insertion_number'] = points_df_with_insertion.groupby('sample_id').cumcount() + 1
    max_insertions = points_df_with_insertion['insertion_number'].max()
else:
    points_df_with_insertion = points_df
    max_insertions = points_df['insertion_number'].max()

# Show info about sample_id distribution
print("\nSample ID information:")
print(f"Total number of unique samples: {points_df_with_insertion['sample_id'].nunique()}")
print(f"Sample ID range: {points_df_with_insertion['sample_id'].min()} to {points_df_with_insertion['sample_id'].max()}")
print(f"Total data points: {len(points_df_with_insertion)}")

# Show sample distribution by insertion number
sample_counts = points_df_with_insertion.groupby('insertion_number')['sample_id'].nunique().reset_index()
sample_counts.columns = ['insertion_number', 'unique_samples']
print("\nSample distribution by insertion number:")
for _, row in sample_counts.iterrows():
    print(f"  Insertion {int(row['insertion_number'])}: {int(row['unique_samples'])} unique samples")

for insertion_num in range(1, max_insertions + 1):
    # Get data for this insertion number
    insertion_data = points_df_with_insertion[points_df_with_insertion['insertion_number'] == insertion_num]
    
    if len(insertion_data) > 0:
        x_values = insertion_data['x'].to_numpy()
        y_values = insertion_data['y'].to_numpy()
        sample_ids = insertion_data['sample_id'].to_numpy()
        
        # Filter out invalid points
        valid_mask = np.isfinite(x_values) & np.isfinite(y_values) & (x_values != 0)
        x_valid = x_values[valid_mask]
        y_valid = y_values[valid_mask]
        sample_ids_valid = sample_ids[valid_mask]
        
        if len(x_valid) > 0:
            x_mean = np.mean(x_valid)
            x_std = np.std(x_valid)
            y_mean = np.mean(y_valid)
            y_std = np.std(y_valid)
            
            insertion_stats.append({
                'insertion_number': insertion_num,
                'x_mean': x_mean,
                'x_std': x_std,
                'y_mean': y_mean,
                'y_std': y_std,
                'n_points': len(x_valid),
                'unique_samples': len(np.unique(sample_ids_valid)),
                'sample_ids': list(np.unique(sample_ids_valid))
            })

# Convert to DataFrame for easier handling
stats_df = pd.DataFrame(insertion_stats)

if len(stats_df) > 0:
    # Create the plotly figure
    fig = go.Figure()

    # Add scatter plot with error bars (only y-axis std)
    fig.add_trace(go.Scatter(
        x=stats_df['x_mean'],
        y=stats_df['y_mean'],
        error_y=dict(type='data', array=stats_df['y_std'], visible=True),
        mode='markers+text',
        text=[str(int(row['insertion_number'])) for _, row in stats_df.iterrows()],
        textposition='top right',
        textfont=dict(size=10),
        marker=dict(size=8, opacity=0.7),
        name=f'Mean impact by insertion (n={len(stats_df)})',
        hovertemplate='Insertion: %{text}<br>' +
                      'x_mean: %{x:.6f}<br>' +
                      'y_mean: %{y:.6f}<br>' +
                      '<extra></extra>'
    ))

    # Add fitted line if we have enough points
    if len(stats_df) >= 2:
        x_mean_valid = stats_df['x_mean'].to_numpy()
        y_mean_valid = stats_df['y_mean'].to_numpy()
        
        # Fit line to mean values
        beta_insertion = beta_fit(x_mean_valid, y_mean_valid, alpha_global, method=est_method)
        
        if np.isfinite(beta_insertion):
            x_range = np.linspace(x_mean_valid.min(), x_mean_valid.max(), 100)
            y_fitted = alpha_global + beta_insertion * x_range
            fig.add_trace(go.Scatter(
                x=x_range,
                y=y_fitted,
                mode='lines',
                line=dict(color='red', dash='dash', width=2),
                name=f'Fitted: β = {beta_insertion:.6f}',
                hovertemplate='Fitted line<br>' +
                              'x: %{x:.6f}<br>' +
                              'y: %{y:.6f}<br>' +
                              '<extra></extra>'
            ))

    # Add theoretical line
    beta_theory = 0.5
    x_range_theory = np.linspace(stats_df['x_mean'].min(), stats_df['x_mean'].max(), 100)
    y_theory = alpha_global + beta_theory * x_range_theory
    fig.add_trace(go.Scatter(
        x=x_range_theory,
        y=y_theory,
        mode='lines',
        line=dict(color='green', dash='dot', width=2),
        name=f'Theoretical: β = {beta_theory:.6f}',
        hovertemplate='Theoretical line<br>' +
                      'x: %{x:.6f}<br>' +
                      'y: %{y:.6f}<br>' +
                      '<extra></extra>'
    ))

    # Update layout
    fig.update_layout(
        title='Market Impact: Mean by Insertion Number',
        xaxis_title='log(Q / V_exp) - Mean',
        yaxis_title='log(Impact) - Mean',
        width=1000,
        height=600,
        showlegend=True,
        hovermode='closest'
    )

    # Add grid
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

    # Display the plot
    fig.show()

    # Print summary statistics
    print(f"\nSummary Statistics:")
    print(f"Number of insertions analyzed: {len(stats_df)}")
    print(f"Alpha (fixed intercept): {alpha_global:.6f}")
    if len(stats_df) >= 2 and 'beta_insertion' in locals() and np.isfinite(beta_insertion):
        print(f"Fitted Beta (insertion means): {beta_insertion:.6f}")
    print(f"Theoretical Beta: {beta_theory:.6f}")
    print(f"Maximum insertion number: {max_insertions}")
    
    # Print detailed insertion statistics with sample info
    print(f"\nDetailed insertion statistics:")
    for _, row in stats_df.iterrows():
        print(f"  Insertion {int(row['insertion_number'])}: {int(row['n_points'])} points from {int(row['unique_samples'])} unique samples")
        print(f"    Sample IDs: {sorted(row['sample_ids'])}")
        print(f"    x_mean: {row['x_mean']:.6f} ± {row['x_std']:.6f}")
        print(f"    y_mean: {row['y_mean']:.6f} ± {row['y_std']:.6f}")
else:
    print("No valid insertion data found for analysis")


Available columns in points_df: ['sample_id', 'insertion', 'x', 'y']
Using sample-based analysis instead...

Sample ID information:
Total number of unique samples: 168
Sample ID range: 256 to 1006177
Total data points: 1316

Sample distribution by insertion number:
  Insertion 1: 168 unique samples
  Insertion 2: 164 unique samples
  Insertion 3: 162 unique samples
  Insertion 4: 161 unique samples
  Insertion 5: 158 unique samples
  Insertion 6: 150 unique samples
  Insertion 7: 141 unique samples
  Insertion 8: 126 unique samples
  Insertion 9: 86 unique samples



Summary Statistics:
Number of insertions analyzed: 9
Alpha (fixed intercept): -3.195274
Fitted Beta (insertion means): 0.645155
Theoretical Beta: 0.500000
Maximum insertion number: 9

Detailed insertion statistics:
  Insertion 1: 168 points from 168 unique samples
    Sample IDs: [256, 257, 373, 441, 512, 545, 557, 585, 819, 844, 850, 972, 1004, 1034, 1074, 1102, 1123, 1151, 1176, 1219, 1232, 1239, 1296, 1340, 1448, 1548, 1555, 1579, 1609, 1673, 1753, 1899, 1962, 2046, 2131, 2158, 2213, 2287, 2302, 2432, 2569, 2664, 2678, 2802, 2858, 2870, 2956, 3040, 3144, 3263, 3373, 3401, 3463, 3651, 3722, 3742, 3765, 3778, 3879, 3931, 4105, 4497, 4643, 4694, 4733, 4850, 5073, 5198, 5200, 5263, 5416, 5461, 5494, 5530, 5560, 5586, 5619, 5687, 5725, 5939, 5972, 6063, 6177, 1000037, 1000256, 1000257, 1000373, 1000441, 1000486, 1000545, 1000557, 1000844, 1000850, 1000972, 1001004, 1001034, 1001074, 1001102, 1001123, 1001151, 1001176, 1001219, 1001232, 1001239, 1001340, 1001448, 1001548, 1001555, 100157

In [18]:
# Calculate mean and std for each x-axis bin across all samples
import numpy as np

# First, let's check what columns are available in points_df
print("Available columns in points_df:", points_df.columns.tolist())

# Get all valid x and y values
x_values = points_df['x'].to_numpy()
y_values = points_df['y'].to_numpy()

# Filter out invalid points
valid_mask = np.isfinite(x_values) & np.isfinite(y_values) & (x_values != 0)
x_valid = x_values[valid_mask]
y_valid = y_values[valid_mask]

if len(x_valid) > 0:
    # Create bins for x-axis
    n_bins = 1000  # Number of bins
    x_min, x_max = x_valid.min(), x_valid.max()
    bin_edges = np.linspace(x_min, x_max, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Calculate statistics for each bin
    bin_stats = []
    
    for i in range(n_bins):
        # Find points in this bin
        bin_mask = (x_valid >= bin_edges[i]) & (x_valid < bin_edges[i + 1])
        if i == n_bins - 1:  # Include the last edge in the final bin
            bin_mask = (x_valid >= bin_edges[i]) & (x_valid <= bin_edges[i + 1])
        
        x_bin = x_valid[bin_mask]
        y_bin = y_valid[bin_mask]
        
        if len(x_bin) > 0:
            x_mean = np.mean(x_bin)
            x_std = np.std(x_bin)
            y_mean = np.mean(y_bin)
            y_std = np.std(y_bin)
            
            bin_stats.append({
                'bin_number': i + 1,
                'bin_center': bin_centers[i],
                'x_mean': x_mean,
                'x_std': x_std,
                'y_mean': y_mean,
                'y_std': y_std,
                'n_points': len(x_bin),
                'x_min': x_bin.min(),
                'x_max': x_bin.max()
            })

    # Convert to DataFrame for easier handling
    stats_df = pd.DataFrame(bin_stats)

    if len(stats_df) > 0:
        # Create the plotly figure
        fig = go.Figure()

        # Add scatter plot with error bars (only y-axis std)
        fig.add_trace(go.Scatter(
            x=stats_df['x_mean'],
            y=stats_df['y_mean'],
            error_y=dict(type='data', array=stats_df['y_std'], visible=True, color='blue'),
            mode='markers',
            marker=dict(size=5, color='black', opacity=1.0),
            name=f'Mean impact by x-bin (n={len(stats_df)})',
            hovertemplate='Bin: %{customdata}<br>' +
                          'x_mean: %{x:.6f}<br>' +
                          'y_mean: %{y:.6f}<br>' +
                          'n_points: %{customdata}<br>' +
                          '<extra></extra>',
            customdata=stats_df['bin_number']
        ))

        # Add fitted line if we have enough points
        if len(stats_df) >= 2:
            x_mean_valid = stats_df['x_mean'].to_numpy()
            y_mean_valid = stats_df['y_mean'].to_numpy()
            
            # Fit line to mean values
            beta_bins = beta_fit(x_mean_valid, y_mean_valid, alpha_global, method=est_method)
            
            if np.isfinite(beta_bins):
                x_range = np.linspace(x_mean_valid.min(), x_mean_valid.max(), 100)
                y_fitted = alpha_global + beta_bins * x_range
                fig.add_trace(go.Scatter(
                    x=x_range,
                    y=y_fitted,
                    mode='lines',
                    line=dict(color='red', dash='dash', width=2),
                    name=f'Fitted: β = {beta_bins:.6f}',
                    hovertemplate='Fitted line<br>' +
                                  'x: %{x:.6f}<br>' +
                                  'y: %{y:.6f}<br>' +
                                  '<extra></extra>'
                ))

        # Add theoretical line
        beta_theory = 0.5
        x_range_theory = np.linspace(stats_df['x_mean'].min(), stats_df['x_mean'].max(), 100)
        y_theory = alpha_global + beta_theory * x_range_theory
        fig.add_trace(go.Scatter(
            x=x_range_theory,
            y=y_theory,
            mode='lines',
            line=dict(color='green', dash='dot', width=2),
            name=f'Theoretical: β = {beta_theory:.6f}',
            hovertemplate='Theoretical line<br>' +
                          'x: %{x:.6f}<br>' +
                          'y: %{y:.6f}<br>' +
                          '<extra></extra>'
        ))

        # Update layout
        fig.update_layout(
            title='Market Impact: Mean by X-Axis Bins',
            xaxis_title='log(Q / V_exp) - Mean',
            yaxis_title='log(Impact) - Mean',
            width=1000,
            height=600,
            showlegend=True,
            hovermode='closest'
        )

        # Add grid
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

        # Display the plot
        fig.show()

        # Print summary statistics
        print(f"Number of x-bins analyzed: {len(stats_df)}")
        print(f"Number of bins: {n_bins}")
        print(f"Total points used: {len(x_valid)}")
        print(f"Alpha (fixed intercept): {alpha_global:.6f}")
        if len(stats_df) >= 2 and 'beta_bins' in locals() and np.isfinite(beta_bins):
            print(f"Fitted Beta (x-bin means): {beta_bins:.6f}")
        print(f"Theoretical Beta: {beta_theory:.6f}")
        print(f"X-axis range: [{x_min:.6f}, {x_max:.6f}]")
        
        # Print bin details
        print("\nBin details:")
        for _, row in stats_df.iterrows():
            print(f"Bin {int(row['bin_number'])}: x=[{row['x_min']:.6f}, {row['x_max']:.6f}], "
                  f"x_mean={row['x_mean']:.6f}, y_mean={row['y_mean']:.6f}, n={int(row['n_points'])}")
    else:
        print("No valid bin data found for analysis")
else:
    print("No valid data points found for analysis")


Available columns in points_df: ['sample_id', 'insertion', 'x', 'y']


Number of x-bins analyzed: 432
Number of bins: 1000
Total points used: 1316
Alpha (fixed intercept): -3.195274
Fitted Beta (x-bin means): 0.631215
Theoretical Beta: 0.500000
X-axis range: [-11.553077, -8.272200]

Bin details:
Bin 1: x=[-11.553077, -11.553077], x_mean=-11.553077, y_mean=-10.292783, n=1
Bin 34: x=[-11.441851, -11.441851], x_mean=-11.441851, y_mean=-8.799052, n=1
Bin 142: x=[-11.090453, -11.090453], x_mean=-11.090453, y_mean=-8.809496, n=1
Bin 254: x=[-10.722728, -10.722728], x_mean=-10.722728, y_mean=-10.052901, n=2
Bin 257: x=[-10.709989, -10.709989], x_mean=-10.709989, y_mean=-8.422970, n=1
Bin 265: x=[-10.684988, -10.684988], x_mean=-10.684988, y_mean=-9.445389, n=2
Bin 273: x=[-10.660597, -10.660597], x_mean=-10.660597, y_mean=-9.166783, n=1
Bin 280: x=[-10.636786, -10.636786], x_mean=-10.636786, y_mean=-7.800129, n=1
Bin 283: x=[-10.625090, -10.625090], x_mean=-10.625090, y_mean=-9.201248, n=1
Bin 294: x=[-10.590801, -10.590801], x_mean=-10.590801, y_mean=-9.240386,

# Save data needed for mega plot

In [19]:
import pickle
import os

# Define the experiment folder (replace with your actual experiment folder variable if needed)
experiment_folder = os.getcwd()  # or set to your experiment folder path

# Save all_betas_evo
with open(os.path.join(experiment_folder, "all_betas_evo.pkl"), "wb") as f:
    pickle.dump(all_betas_evo, f)

# Save insertion_data_dict
with open(os.path.join(experiment_folder, "insertion_data_dict.pkl"), "wb") as f:
    pickle.dump(insertion_data_dict, f)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:9                                                                                    │
│                                                                                                  │
│    6                                                                                             │
│    7 # Save all_betas_evo                                                                        │
│    8 with open(os.path.join(experiment_folder, "all_betas_evo.pkl"), "wb") as f:                 │
│ ❱  9 │   pickle.dump(all_betas_evo, f)                                                           │
│   10                                                                                             │
│   11 # Save insertion_data_dict                                                                  │
│   12 with open(os.path.join(experiment_folder, "insertion_data_dict.pkl"), "wb") as f:           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'all_betas_evo' is not defined

In [ ]:
experiment_folder